# MDM DJBC - Master Data Management

## Single Importer & Exporter View

**Kelompok 5**

Notebook ini mengimplementasikan pipeline Master Data Management (MDM) untuk
mengintegrasikan data dari OSS (Online Single Submission) dan CEISA (Customs
Excise Information System) menjadi Golden Record tunggal per entitas perusahaan.

### Tahapan:
1. **Tahap 0** - Simulasi Data & Faker Engine
2. **Tahap 1** - Data Profiling
3. **Tahap 2** - Data Cleansing & Standardization
4. **Tahap 3** - Duplicate Detection & Matching
5. **Tahap 4** - Golden Record & Survivorship
6. **Tahap 5** - Data Quality Monitoring
7. **Tahap 6** - YData Profiling Dashboard

---

### Narasi Pembuka

**Master Data Management (MDM)** adalah pendekatan strategis untuk mengelola data master
perusahaan. Dalam konteks DJBC, data importir dan eksportir nasional tersebar di dua
sistem utama: **OSS** (sistem perizinan berusaha) dan **CEISA** (sistem operasional
kepabeanan).

Permasalahan utama yang dihadapi:
1. Data tersebar di dua sistem yang tidak terintegrasi
2. Ketidakseragaman format data antar sistem
3. Duplikasi data intra-sistem maupun antar-sistem
4. Konflik data (status berbeda, data usang, dll.)

Pipeline MDM yang diimplementasikan mencakup simulasi data dengan anomali sengaja,
profiling kualitas data, pembersihan dan standardisasi, pencocokan antar sumber,
pembentukan Golden Record, monitoring kualitas, dan dashboard profiling akhir.

> **Catatan:** Notebook ini dirancang untuk Google Colab. Semua library akan diinstall
> otomatis. Data akan disimpan di folder `/content/`.

## Setup Environment

### 1. Instalasi Library

Cell pertama menginstall semua library yang dibutuhkan:

| Library | Fungsi |
|---------|--------|
| missingno | Visualisasi missing values |
| faker | Generator data simulasi |
| ydata-profiling | Laporan profiling otomatis |
| fuzzywuzzy, jellyfish | Fuzzy string matching |
| recordlinkage | Framework record linkage |
| networkx | Analisis graf untuk clustering |

In [ ]:
# Install library untuk Google Colab
import warnings
warnings.filterwarnings('ignore')

!pip install missingno faker ydata-profiling fuzzywuzzy python-Levenshtein jellyfish recordlinkage networkx -q

print('Semua library berhasil diinstall!')

### 2. Import Semua Library

Mengimport library yang akan digunakan di seluruh pipeline.

In [ ]:
# ============================================================
# CELL 1: INSTALL LIBRARY TAMBAHAN
# Jalankan cell ini pertama kali sebelum cell lainnya
# Estimasi waktu: 30-60 detik
# ============================================================

# Install missingno untuk visualisasi missing values
import subprocess
# subprocess.check_call(['pip', 'install', 'missingno', '-q'])  # Already installed in previous cell

# Install faker untuk generate data simulasi
# subprocess.check_call(['pip', 'install', 'faker', '-q'])  # Already installed in previous cell

# Install ydata-profiling untuk laporan profiling Tahap 1 & Tahap 6
# subprocess.check_call(['pip', 'install', 'ydata-profiling', '-q'])  # Already installed in previous cell

# Konfirmasi instalasi berhasil
print('✅ Instalasi library selesai!')
print('Library yang tersedia:')
print('  - pandas       : manipulasi dan analisis data')
print('  - numpy        : komputasi numerik')
print('  - faker        : generate data simulasi realistis')
print('  - matplotlib   : visualisasi dasar')
print('  - seaborn      : visualisasi statistik')
print('  - missingno    : visualisasi missing values')


# ============================================================
# CELL 2: IMPORT SEMUA LIBRARY
# ============================================================

import pandas as pd               # manipulasi dataframe
import numpy as np                # komputasi numerik
import re                         # regular expression untuk validasi format
import random                     # random number generator
import warnings
warnings.filterwarnings('ignore') # sembunyikan warning yang tidak penting

# Faker untuk generate data simulasi
from faker import Faker
from faker.providers import person, address, company, internet, phone_number

# Visualisasi
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno

# Setting tampilan
pd.set_option('display.max_columns', None)     # tampilkan semua kolom
pd.set_option('display.max_rows', 50)          # max 50 baris ditampilkan
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 desimal
pd.set_option('display.width', 120)

# Setting style visualisasi
sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family']    = 'sans-serif'

# Set seed agar data yang dihasilkan konsisten
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('✅ Semua library berhasil diimport!')
print(f'   Pandas versi  : {pd.__version__}')
print(f'   NumPy versi   : {np.__version__}')



### 3. Setup Direktori Google Colab

Membuat struktur folder untuk menyimpan data dan laporan.

In [ ]:
# Setup base path untuk Google Colab
import os

BASE_PATH = '/content'
DATA_RAW = os.path.join(BASE_PATH, 'data/raw')
DATA_PROCESSED = os.path.join(BASE_PATH, 'data/processed')
DATA_GOLDEN = os.path.join(BASE_PATH, 'data/golden')
REPORTS_DIR = os.path.join(BASE_PATH, 'reports')

for d in [DATA_RAW, DATA_PROCESSED, DATA_GOLDEN, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

print('Struktur folder siap:')
print(f'  Data mentah   : {DATA_RAW}/')
print(f'  Data bersih   : {DATA_PROCESSED}/')
print(f'  Golden Record : {DATA_GOLDEN}/')
print(f'  Laporan       : {REPORTS_DIR}/')

# Untuk menyimpan permanen di Google Drive, gunakan:
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_PATH = '/content/drive/MyDrive/MDM_DJBC'

# TAHAP 0: SIMULASI DATA & FAKER ENGINE

**Tujuan:** Membangkitkan dataset simulasi yang mencerminkan kompleksitas dunia nyata
di DJBC, lengkap dengan anomali untuk menguji sistem MDM.

### Penjelasan

Tahap ini membangkitkan data untuk dua sistem:
1. **OSS** - Sistem perizinan berusaha (System of Record untuk legalitas)
2. **CEISA** - Sistem operasional kepabeanan

**Parameter:** 5.000 perusahaan unit, ~6.000-6.500 total record.

**Anomali yang disuntikkan:**
- NPWP kotor (tanpa separator) di CEISA (25%)
- Typo nama perusahaan (15%)
- NIB invalid (15%)
- Konflik status OSS vs CEISA (200 record)
- Logical conflict (FLAG_EKSPOR=N vs NIPER terisi)
- Duplicate entry OSS (10%)
- Data mart snapshot CEISA (5%)
- Stale data (3%)
- Missing optional fields (15%)

### Lab 0.1 - Data Referensi DJBC

Kode referensi standar DJBC: provinsi, KPPBC, enum values.

In [ ]:
# [LAB 0.1] DATA REFERENSI DJBC
# Berisi kode-kode referensi standar Direktorat Jenderal Bea dan Cukai (DJBC)
# sesuai dengan docs/01_data_dictionary.md

# Referensi Provinsi Indonesia (34 provinsi)
PROVINSI = {
    '11': 'Aceh',               '12': 'Sumatera Utara',
    '13': 'Sumatera Barat',      '14': 'Riau',
    '15': 'Jambi',               '16': 'Sumatera Selatan',
    '17': 'Bengkulu',            '18': 'Lampung',
    '19': 'Kep. Bangka Belitung','21': 'Kep. Riau',
    '31': 'DKI Jakarta',         '32': 'Jawa Barat',
    '33': 'Jawa Tengah',         '34': 'DI Yogyakarta',
    '35': 'Jawa Timur',          '36': 'Banten',
    '51': 'Bali',                '52': 'Nusa Tenggara Barat',
    '53': 'Nusa Tenggara Timur', '61': 'Kalimantan Barat',
    '62': 'Kalimantan Tengah',   '63': 'Kalimantan Selatan',
    '64': 'Kalimantan Timur',    '65': 'Kalimantan Utara',
    '71': 'Sulawesi Utara',      '72': 'Sulawesi Tengah',
    '73': 'Sulawesi Selatan',    '74': 'Sulawesi Tenggara',
    '75': 'Gorontalo',           '76': 'Sulawesi Barat',
    '81': 'Maluku',              '82': 'Maluku Utara',
    '91': 'Papua Barat',         '94': 'Papua',
}

# Daftar Kantor Pelayanan Bea Cukai (KPPBC)
# KODE_KANTOR yang digunakan dalam OSS & CEISA
KPPBC_LIST = [
    # Kantor Utama / Besar (sudah ada di list asli)
    {'kode': '010100', 'nama': 'KPU Bea dan Cukai Tipe A Tanjung Priok', 'wilayah': '31'},
    {'kode': '040300', 'nama': 'KPPBC TMP B Soekarno-Hatta',           'wilayah': '31'},
    {'kode': '020300', 'nama': 'KPPBC TMP Belawan',                   'wilayah': '12'},
    {'kode': '070100', 'nama': 'KPPBC TMP Tanjung Perak',             'wilayah': '35'},
    {'kode': '050100', 'nama': 'KPPBC TMP Tanjung Emas',              'wilayah': '33'},
    {'kode': '140100', 'nama': 'KPPBC TMP B Makassar',                'wilayah': '73'},
    {'kode': '060100', 'nama': 'KPPBC TMP A Pasuruan',                'wilayah': '35'},
    {'kode': '090100', 'nama': 'KPPBC TMP B Ngurah Rai',              'wilayah': '51'},

    # Tambahan utama & representatif dari berbagai wilayah
    {'kode': '010700', 'nama': 'KPPBC Tipe Madya Pabean Belawan',     'wilayah': '12'},
    {'kode': '010800', 'nama': 'KPPBC Tipe Madya Pabean B Medan',     'wilayah': '12'},
    {'kode': '011200', 'nama': 'KPPBC Tipe Madya Pabean C Kuala Tanjung', 'wilayah': '12'},
    {'kode': '020400', 'nama': 'KPU Bea dan Cukai Tipe B Batam',      'wilayah': '21'},
    {'kode': '020100', 'nama': 'KPPBC Tipe Madya Pabean B Tanjung Balai Karimun', 'wilayah': '21'},
    {'kode': '030100', 'nama': 'KPPBC Tipe Madya Pabean B Palembang', 'wilayah': '16'},
    {'kode': '030700', 'nama': 'KPPBC Tipe Madya Pabean B Bandar Lampung', 'wilayah': '18'},
    {'kode': '050400', 'nama': 'KPPBC Tipe Madya Pabean Merak',       'wilayah': '36'},
    {'kode': '050900', 'nama': 'KPPBC Tipe Madya Pabean A Bekasi',    'wilayah': '32'},
    {'kode': '070500', 'nama': 'KPPBC TMP Juanda',                    'wilayah': '35'},
    {'kode': '080100', 'nama': 'KPPBC TMP Ngurah Rai',                'wilayah': '51'},
    {'kode': '100300', 'nama': 'KPPBC Balikpapan',                    'wilayah': '64'},
    {'kode': '110100', 'nama': 'KPPBC Makassar',                      'wilayah': '73'},
    {'kode': '120300', 'nama': 'KPPBC Sorong',                        'wilayah': '91'},
    {'kode': '040400', 'nama': 'KPPBC Tipe Madya Pabean A Jakarta',   'wilayah': '31'},
    {'kode': '060300', 'nama': 'KPPBC Tipe Madya Cukai Kudus',        'wilayah': '33'},
    {'kode': '071300', 'nama': 'KPPBC Pasuruan',                      'wilayah': '35'},
    {'kode': '090400', 'nama': 'KPPBC Pontianak',                     'wilayah': '61'},
]

# --- POOLS UNTUK SIMULASI (ENUMS) ---

# Status NIB (Sesuai docs/01_data_dictionary.md §1.16)
STATUS_NIB_POOL = ['AKTIF', 'DIBEKUKAN', 'DICABUT']

# Status Badan Hukum (Sesuai docs/01_data_dictionary.md §1.6)
STATUS_BADAN_HUKUM_POOL = ['Berbadan Hukum', 'Belum Berbadan Hukum']

# Status Perseroan (Sesuai docs/01_data_dictionary.md §1.7)
STATUS_PERSEROAN_POOL = ['Aktif', 'Tidak Aktif', 'Dibekukan']

# Jenis API (Angka Pengenal Importir) (Sesuai docs/01_data_dictionary.md §1.14)
JENIS_API_POOL = ['API-U', 'API-P']

# Kategori Pelaku Usaha di CEISA (Sesuai docs/01_data_dictionary.md §2.10)
KATEGORI_CEISA_POOL = ['IMPORTIR', 'EKSPORTIR', 'KEDUA-DUANYA']

# Jenis Badan Usaha (Sesuai docs/01_data_dictionary.md §1.5)
JENIS_PERSEROAN_POOL = ['PT', 'CV', 'Firma', 'Perum', 'UD']

# Flag Fasilitas (Y/N)
FLAG_POOL = ['Y', 'N']

print(f'✅ Lab 0.1: Referensi DJBC Siap (Align dengan Data Dictionary)')
print(f'   - {len(KPPBC_LIST)} Kantor KPPBC terdaftar.')
print(f'   - {len(STATUS_NIB_POOL)} Status NIB didefinisikan.')


### Lab 0.2 - Generator Identifier DJBC

Fungsi untuk menghasilkan NIB, NPWP, API, NIPER, dan tanggal.

In [ ]:
# [LAB 0.2] GENERATOR IDENTIFIER DJBC
import random
from datetime import datetime, timedelta

def generate_nib_valid():
    """Generate NIB 13 digit numerik sesuai standar OSS"""
    return ''.join([str(random.randint(0, 9)) for _ in range(13)])

def generate_nib_invalid():
    """Generate NIB bermasalah untuk simulasi anomali"""
    error_types = [
        lambda: ''.join([str(random.randint(0, 9)) for _ in range(12)]), # Kurang digit
        lambda: ''.join([str(random.randint(0, 9)) for _ in range(13)]) + 'X', # Ada huruf
        lambda: '0000000000000' # Dummy/Kosong
    ]
    return random.choice(error_types)()

def generate_npwp_valid():
    """Generate NPWP format baku: XX.XXX.XXX.X-XXX.XXX (15 digit)"""
    d = [str(random.randint(0,99)).zfill(2), str(random.randint(0,999)).zfill(3), 
         str(random.randint(0,999)).zfill(3), str(random.randint(0,9)),
         str(random.randint(0,999)).zfill(3), str(random.randint(0,999)).zfill(3)]
    return f"{d[0]}.{d[1]}.{d[2]}.{d[3]}-{d[4]}.{d[5]}"

def generate_npwp_invalid():
    """Generate NPWP kotor (tanpa separator/salah format)"""
    raw_15 = ''.join([str(random.randint(0, 9)) for _ in range(15)])
    error_types = [
        lambda: raw_15, # Tanpa titik/strip (sering di CEISA)
        lambda: f"{raw_15[:9]}", # Digit kurang
        lambda: raw_15.replace('0', 'O').replace('1', 'I'), # Typo karakter mirip
        lambda: f"{raw_15[:2]} {raw_15[2:5]} {raw_15[5:8]}" # Pake spasi
    ]
    return random.choice(error_types)()

def generate_api_valid():
    """Generate Nomor API (10 digit numerik)"""
    return ''.join([str(random.randint(0, 9)) for _ in range(10)])

def generate_niper_valid():
    """Generate Nomor NIPER (10 digit numerik)"""
    return ''.join([str(random.randint(0, 9)) for _ in range(10)])

def generate_date_random(start_year=2020, end_year=2025):
    """Generate tanggal acak dalam format string YYYY-MM-DD"""
    start_date = datetime(start_year, 1, 1)
    end_date = datetime(end_year, 12, 31)
    time_between_dates = end_date - start_date
    days_between_dates = time_between_dates.days
    random_days = random.randrange(days_between_dates)
    return (start_date + timedelta(days=random_days)).strftime('%Y-%m-%d')

def generate_date_recent(max_days_ago=90):
    """Generate tanggal acak dalam N hari terakhir dari hari ini (format YYYY-MM-DD)"""
    days_ago = random.randint(0, max_days_ago)
    return (datetime.now() - timedelta(days=days_ago)).strftime('%Y-%m-%d')


### Lab 1 - DJBC Data Simulation Engine

Fungsi utama simulasi yang membangkitkan data OSS dan CEISA dengan anomali.

In [ ]:
# [LAB 1] DJBC DATA SIMULATION ENGINE
# Berdasarkan: docs/01_data_dictionary.md & docs/02_business_rules.md

import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime
# References already defined above
# Generators already defined above

# Inisialisasi
fake = Faker('id_ID')
Faker.seed(42)
random.seed(42)

N_MASTER = 5000  # Total perusahaan unik

def run_full_simulation():
    print(f"🚀 Memulai simulasi {N_MASTER} perusahaan (DIRTY MODE)... ")

    # 1. BANGKITKAN MASTER ENTITIES (OSS as Truth) - 16 kolom sesuai data dictionary
    #    KODE_KANTOR ikut dibawa sebagai helper untuk CEISA, di-drop sebelum export OSS
    master_list = []
    for i in range(N_MASTER):
        # 15% NIB Invalid di Master (Validity NIB)
        if random.random() < 0.15:
            nib = generate_nib_invalid()
        else:
            nib = generate_nib_valid()

        npwp = generate_npwp_valid()
        nama = fake.company().upper()
        kppbc = random.choice(KPPBC_LIST)

        flag_impor = random.choice(['Y', 'N'])
        jenis_api = random.choice(JENIS_API_POOL) if flag_impor == 'Y' else ""

        master_list.append({
            'NIB': nib,
            'NPWP_PERSEROAN': npwp,
            'NAMA_PERSEROAN': nama,
            'NAMA_SINGKATAN': nama.split()[0][:5],
            'JENIS_PERSEROAN': random.choice(JENIS_PERSEROAN_POOL),
            'STATUS_BADAN_HUKUM': random.choice(STATUS_BADAN_HUKUM_POOL),
            'STATUS_PERSEROAN': random.choice(STATUS_PERSEROAN_POOL),
            'ALAMAT_PERSEROAN': fake.street_address().upper(),
            'KELURAHAN_PERSEROAN': fake.city().upper(),
            'PERSEROAN_DAERAH_ID': kppbc['wilayah'],
            'KODE_POS_PERSEROAN': fake.postcode(),
            'FLAG_IMPOR': flag_impor,
            'FLAG_EKSPOR': random.choice(['Y', 'N']),
            'JENIS_API': jenis_api,
            'TGL_PERUBAHAN_NIB': generate_date_random(2023, 2025),
            'STATUS_NIB': random.choices(STATUS_NIB_POOL, weights=[80, 10, 10])[0],
            'KODE_KANTOR': kppbc['kode'],
        })

    df_oss = pd.DataFrame(master_list)

    # --- INJEKSI KEKOTORAN GLOBAL (HEAVY ATTACK) ---
    # 1. Completeness: hanya field opsional yang dikosongkan ("Missing Optional Field").
    #    Field wajib (NIB, NPWP, NAMA, STATUS_NIB) dijamin 100% terisi sesuai business_rules §1.
    for col in ['KELURAHAN_PERSEROAN', 'KODE_POS_PERSEROAN', 'NAMA_SINGKATAN']:
        idx_null = df_oss.sample(frac=0.15).index
        df_oss.loc[idx_null, col] = np.nan

    # 2. Stale Data: sebagian record TGL_PERUBAHAN_NIB sangat lama -> target IS_STALE
    idx_stale_date = df_oss.sample(frac=0.03).index
    df_oss.loc[idx_stale_date, 'TGL_PERUBAHAN_NIB'] = [generate_date_random(2014, 2015) for _ in range(len(idx_stale_date))]

    # 3. Uniqueness: tambah 10% record duplikat persis ("Duplicate Entry")
    df_dups = df_oss.sample(frac=0.10)
    df_oss = pd.concat([df_oss, df_dups], ignore_index=True)

    # 2. CREATE CEISA DATASET (Subset & Heavy Anomaly) - 16 kolom sesuai data dictionary
    df_ceisa = df_oss.sample(frac=0.9).copy()

    # Transformasi kolom: OSS -> CEISA
    df_ceisa = df_ceisa.rename(columns={
        'NPWP_PERSEROAN': 'NPWP',
        'NAMA_PERSEROAN': 'NAMA_PERUSAHAAN',
        'ALAMAT_PERSEROAN': 'ALAMAT_PERUSAHAAN',
        'KELURAHAN_PERSEROAN': 'KELURAHAN',
        'PERSEROAN_DAERAH_ID': 'DAERAH_ID',
        'KODE_POS_PERSEROAN': 'KODE_POS',
        'TGL_PERUBAHAN_NIB': 'TGL_TERBIT_NIB'
    })

    df_ceisa['ID_PERUSAHAAN'] = [f"C{str(i).zfill(6)}" for i in range(len(df_ceisa))]
    df_ceisa['NOMOR_TELPON'] = [fake.phone_number() for _ in range(len(df_ceisa))]
    df_ceisa['KATEGORI'] = random.choices(KATEGORI_CEISA_POOL, k=len(df_ceisa))
    df_ceisa['NIPER'] = [generate_niper_valid() if k != 'IMPORTIR' else "" for k in df_ceisa['KATEGORI']]
    df_ceisa['NOMOR_API'] = [generate_api_valid() if k != 'EKSPORTIR' else "" for k in df_ceisa['KATEGORI']]
    # TGL_SYNC_OSS: tanggal sync terakhir dari OSS - dasar HIGH_SYNC_LAG & data mart dedup
    df_ceisa['TGL_SYNC_OSS'] = [generate_date_recent(90) for _ in range(len(df_ceisa))]

    ceisa_cols = ['ID_PERUSAHAAN', 'NIB', 'NPWP', 'NAMA_PERUSAHAAN', 'ALAMAT_PERUSAHAAN',
                  'KELURAHAN', 'DAERAH_ID', 'KODE_POS', 'NOMOR_TELPON', 'KATEGORI',
                  'NIPER', 'NOMOR_API', 'TGL_TERBIT_NIB', 'STATUS_NIB', 'KODE_KANTOR', 'TGL_SYNC_OSS']
    df_ceisa = df_ceisa[ceisa_cols]

    # 3. INJEKSI ANOMALI BERAT (tahap0_simulation_faker.md §4)
    print("⚠️ Menyuntikkan anomali data tingkat tinggi...")

    # Anomali: Format NPWP - NPWP kotor masif di CEISA (25%)
    idx_npwp = df_ceisa.sample(frac=0.25).index
    df_ceisa.loc[idx_npwp, 'NPWP'] = [generate_npwp_invalid() for _ in range(len(idx_npwp))]

    # Anomali: Typo Nama - fuzzy name di CEISA (15%)
    idx_fuzzy = df_ceisa.sample(frac=0.15).index
    df_ceisa.loc[idx_fuzzy, 'NAMA_PERUSAHAAN'] = df_ceisa.loc[idx_fuzzy, 'NAMA_PERUSAHAAN'].apply(lambda x: str(x).replace("PT ", "") + " (CABANG)")

    # Anomali: Logical Conflict - NIPER (CEISA) terisi padahal FLAG_EKSPOR (OSS) = 'N'
    idx_logic = df_ceisa.sample(frac=0.15).index
    df_ceisa.loc[idx_logic, 'NIPER'] = "4803163678"  # paksa terisi
    df_oss.loc[df_oss.index.isin(idx_logic), 'FLAG_EKSPOR'] = 'N'

    # Anomali: Konflik Status - STATUS_NIB OSS != CEISA -> target IS_OUT_OF_SYNC
    idx_conflict = df_ceisa.sample(n=200).index
    df_ceisa.loc[idx_conflict, 'STATUS_NIB'] = 'AKTIF'
    df_oss.loc[df_oss.index.isin(idx_conflict), 'STATUS_NIB'] = 'DICABUT'

    # Anomali: Data Mart Snapshot Duplicate - NIB sama, snapshot lama dgn NAMA & TGL_SYNC_OSS beda
    idx_snapshot = df_ceisa.sample(frac=0.05).index
    df_snapshot_old = df_ceisa.loc[idx_snapshot].copy()
    df_snapshot_old['NAMA_PERUSAHAAN'] = df_snapshot_old['NAMA_PERUSAHAAN'] + " (OLD)"
    df_snapshot_old['TGL_SYNC_OSS'] = [generate_date_random(2023, 2024) for _ in range(len(df_snapshot_old))]
    df_ceisa = pd.concat([df_ceisa, df_snapshot_old], ignore_index=True)

    # 4. EXPORT DATA
    df_oss = df_oss.drop(columns=['KODE_KANTOR'])  # KODE_KANTOR khusus CEISA sesuai data dictionary
    df_oss.to_csv(f'{DATA_RAW}/oss_nib_data.csv', index=False)
    df_ceisa.to_csv(f'{DATA_RAW}/ceisa_data.csv', index=False)

    print(f"✅ Simulasi Berhasil (DIRTY DATA READY)!")
    print(f"   - OSS: {len(df_oss)} records")
    print(f"   - CEISA: {len(df_ceisa)} records")

# Running simulation...
    run_full_simulation()


# TAHAP 1: DATA PROFILING

**Tujuan:** Eksplorasi dan pengukuran kualitas data baseline sebelum cleansing/MDM.

### Penjelasan

Lingkup analisis per dataset (OSS & CEISA):
1. Dataset overview (shape, dtype, null, unique)
2. Statistik deskriptif (numerik & kategorik)
3. Missing value analysis (severity + visualisasi)
4. Duplicate analysis (full, by NIB, by NAMA)
5. Format validation (NIB, NPWP, KODE_POS, STATUS_NIB)
6. Baseline DQ Score (4 dimensi DMBOK)
7. Laporan YData Profiling

### Fungsi-fungsi Profiling

Fungsi untuk setiap aktivitas profiling yang akan dijalankan.

In [ ]:
# [TAHAP 1] DATA PROFILING
# Berdasarkan: docs/tahapan/tahap1_data_profiling.md & docs/02_business_rules.md §1, §4
#
# Profiling lengkap untuk OSS & CEISA (kondisi sebelum cleansing/MDM):
#   1. Dataset overview (shape, dtype, null, unique)
#   2. Statistik deskriptif (numerik & kategorik)
#   3. Missing value analysis (tabel + severity + visualisasi)
#   4. Duplicate analysis (full duplicate, duplicate by NIB, duplicate by NAMA)
#   5. Format validation (NIB, NPWP, KODE_POS, STATUS_NIB, dst.)
#   6. Baseline Data Quality Score (4 dimensi DMBOK)
#   7. Laporan ydata_profiling -> reports/profiling_before.html

import matplotlib
# # matplotlib.use("Agg")  # commented for Colab  # Not needed in Colab

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import missingno as msno
import re
from datetime import datetime
from ydata_profiling import ProfileReport

# References already defined above
# from Source.step02_reference import (
    STATUS_NIB_POOL, JENIS_PERSEROAN_POOL, KATEGORI_CEISA_POOL,
)

NIB_PATTERN = re.compile(r'^\d{13}$')
NPWP_PATTERN = re.compile(r'^\d{2}\.\d{3}\.\d{3}\.\d{1}-\d{3}\.\d{3}$')
KODE_POS_PATTERN = re.compile(r'^\d{5}$')


def _to_digit_str(series):
    """KODE_POS terbaca float64 (mis. 1330.0) - konversi ke string digit tanpa '.0'
    agar leading zero yang hilang akibat tipe numerik tetap terdeteksi sebagai
    format tidak valid."""
    def conv(x):
        if pd.isna(x):
            return None
        if isinstance(x, float) and x.is_integer():
            return str(int(x))
        return str(x)
    return series.apply(conv)


# ───────────────────────── 1. DATASET OVERVIEW ─────────────────────────
def dataset_overview(df, name):
    print(f"\n{'='*65}\n 1. DATASET OVERVIEW — {name}\n{'='*65}")
    print(f"Jumlah baris : {len(df):,}")
    print(f"Jumlah kolom : {df.shape[1]}")

    overview = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'n_null': df.isnull().sum(),
        'pct_null': (df.isnull().sum() / len(df) * 100).round(2),
        'n_unique': df.nunique(),
    })
    print("\nRingkasan kolom:")
    print(overview.to_string())
    return overview


# ───────────────────────── 2. STATISTIK DESKRIPTIF ─────────────────────────
def descriptive_stats(df, name):
    print(f"\n{'='*65}\n 2. STATISTIK DESKRIPTIF — {name}\n{'='*65}")

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:
        print("\n[Kolom Numerik]")
        print(df[num_cols].describe().to_string())

    print("\n[Kolom Kategorik/Enum - Top 5 Value Counts]")
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    for col in cat_cols:
        if df[col].nunique() <= 20:
            print(f"\n{col}:")
            print(df[col].value_counts(dropna=False).head(5).to_string())


# ───────────────────────── 3. MISSING VALUE ANALYSIS ─────────────────────────
def _severity(pct):
    if pct == 0:
        return 'OK'
    if pct <= 5:
        return 'LOW'
    if pct <= 15:
        return 'MEDIUM'
    if pct <= 30:
        return 'HIGH'
    return 'CRITICAL'


def missing_value_analysis(df, name):
    print(f"\n{'='*65}\n 3. MISSING VALUE ANALYSIS — {name}\n{'='*65}")
    n = len(df)
    n_missing = df.isnull().sum()
    pct_missing = (n_missing / n * 100).round(2)

    table = pd.DataFrame({'n_missing': n_missing, 'pct_missing': pct_missing})
    table['severity'] = table['pct_missing'].apply(_severity)
    table = table[table['n_missing'] > 0].sort_values('pct_missing', ascending=False)

    if table.empty:
        print("Tidak ada missing value pada kolom apapun.")
    else:
        print(table.to_string())

    # Visualisasi: bar chart % missing + missingno matrix
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    pct_missing.sort_values(ascending=False).plot.bar(ax=axes[0], color='#F44336')
    axes[0].set_title(f'% Missing per Kolom — {name}')
    axes[0].set_ylabel('% Missing')
    axes[0].axhline(0, color='black', linewidth=0.5)

    msno.matrix(df, ax=axes[1], sparkline=False)
    axes[1].set_title(f'Missing Value Matrix — {name}')

    plt.tight_layout()
    fname = f'reports/missing_value_{name.lower()}.png'
    plt.savefig(fname, dpi=120, bbox_inches='tight')
    plt.close(fig)
    print(f'\nChart disimpan: {fname}')
    return table


# ───────────────────────── 4. DUPLICATE ANALYSIS ─────────────────────────
def duplicate_analysis(df, name, nama_col):
    print(f"\n{'='*65}\n 4. DUPLICATE ANALYSIS — {name}\n{'='*65}")
    n = len(df)

    full_dup = df.duplicated(keep=False)
    print(f"Full duplicate (semua kolom identik) : {full_dup.sum():,} baris")

    nib_dup = df['NIB'].duplicated(keep=False)
    print(f"Duplicate by NIB                     : {nib_dup.sum():,} baris "
          f"({nib_dup.sum() / n * 100:.2f}%)")

    if name == 'OSS':
        if nib_dup.sum() > 0:
            print("  -> ANOMALI: NIB seharusnya unik di OSS (lihat business rules dimensi 'Unik').")
        else:
            print("  -> OK: NIB unik di OSS, sesuai harapan (sumber legalitas/master).")
    else:
        # CEISA bersifat data mart - duplikat NIB expected, dipisah jadi:
        # - Data Mart Snapshot Duplicate: NIB sama, baris berbeda (snapshot historis)
        # - Duplicate Entry: baris benar-benar identik (anomali)
        snapshot_dup = nib_dup & ~full_dup
        print(f"  -> Data Mart Snapshot Duplicate (NIB sama, isi beda) : {snapshot_dup.sum():,} baris "
              f"(EXPECTED, lihat business rules anomali #7)")
        print(f"  -> Duplicate Entry (baris identik)                   : {full_dup.sum():,} baris "
              f"(ANOMALI, lihat business rules anomali #6)")

    nama_dup = df[nama_col].duplicated(keep=False)
    print(f"Duplicate by {nama_col:<22}: {nama_dup.sum():,} baris ({nama_dup.sum() / n * 100:.2f}%)")

    return {
        'full_duplicate': int(full_dup.sum()),
        'duplicate_by_nib': int(nib_dup.sum()),
        f'duplicate_by_{nama_col}': int(nama_dup.sum()),
    }


# ───────────────────────── 5. FORMAT VALIDATION ─────────────────────────
def format_validation(df, name):
    print(f"\n{'='*65}\n 5. FORMAT VALIDATION — {name}\n{'='*65}")
    n = len(df)
    checks = {}

    nib_valid = df['NIB'].astype(str).str.match(NIB_PATTERN).sum()
    checks['NIB (13 digit)'] = (int(nib_valid), n)

    npwp_col = 'NPWP_PERSEROAN' if name == 'OSS' else 'NPWP'
    npwp_valid = df[npwp_col].astype(str).str.match(NPWP_PATTERN).sum()
    checks[f'{npwp_col} (format XX.XXX.XXX.X-XXX.XXX)'] = (int(npwp_valid), n)

    kp_col = 'KODE_POS_PERSEROAN' if name == 'OSS' else 'KODE_POS'
    kp_series = _to_digit_str(df[kp_col])
    kp_notna = int(kp_series.notna().sum())
    kp_valid = int(kp_series.dropna().str.match(KODE_POS_PATTERN).sum())
    checks[f'{kp_col} (5 digit, dari yg terisi)'] = (kp_valid, kp_notna)

    status_valid = df['STATUS_NIB'].isin(STATUS_NIB_POOL).sum()
    checks['STATUS_NIB (enum AKTIF/DIBEKUKAN/DICABUT)'] = (int(status_valid), n)

    if name == 'OSS':
        jp_valid = df['JENIS_PERSEROAN'].isin(JENIS_PERSEROAN_POOL).sum()
        checks['JENIS_PERSEROAN (enum valid)'] = (int(jp_valid), n)
    else:
        kat_valid = df['KATEGORI'].isin(KATEGORI_CEISA_POOL).sum()
        checks['KATEGORI (enum valid)'] = (int(kat_valid), n)

    for label, (valid, total) in checks.items():
        pct = valid / total * 100 if total else 0.0
        print(f"  {label:<42}: {valid:>5,}/{total:<5,} valid ({pct:6.2f}%)")

    return checks


# ───────────────────────── 6. BASELINE DQ SCORE (4 DIMENSI DMBOK) ─────────────────────────
def calculate_dq_metrics(df, dataset_name="OSS"):
    """
    Menghitung skor kualitas data berdasarkan 4 dimensi DMBOK
    (Kelengkapan, Validitas, Unik, Ketepatan Waktu) sesuai docs/02_business_rules.md §1.
    """
    n_rows = len(df)
    results = {}

    # 1. KELENGKAPAN (Completeness) - field wajib: NIB, NPWP, NAMA, STATUS_NIB
    mand_cols = ['NIB', 'STATUS_NIB']
    mand_cols += ['NPWP_PERSEROAN', 'NAMA_PERSEROAN'] if dataset_name == "OSS" else ['NPWP', 'NAMA_PERUSAHAAN']
    comp_score = (1 - df[mand_cols].isnull().any(axis=1).sum() / n_rows) * 100
    results['Kelengkapan'] = comp_score

    # 2. VALIDITAS (Validity) - format NIB (13 digit) & NPWP (XX.XXX.XXX.X-XXX.XXX)
    v_nib = df['NIB'].astype(str).str.match(NIB_PATTERN).mean() * 100
    npwp_col = 'NPWP_PERSEROAN' if dataset_name == "OSS" else 'NPWP'
    v_npwp = df[npwp_col].astype(str).str.match(NPWP_PATTERN).mean() * 100
    results['Validitas'] = (v_nib + v_npwp) / 2

    # 3. UNIK (Uniqueness) - duplikat NIB
    uniq_score = (df['NIB'].nunique() / n_rows) * 100
    results['Unik'] = uniq_score

    # 4. KETEPATAN WAKTU (Timeliness)
    #    OSS  -> IS_STALE: TGL_PERUBAHAN_NIB > 1 tahun dari sekarang
    #    CEISA -> HIGH_SYNC_LAG: TGL_SYNC_OSS > 30 hari dari sekarang
    if dataset_name == "OSS":
        tgl = pd.to_datetime(df['TGL_PERUBAHAN_NIB'], errors='coerce')
        is_stale = (datetime.now() - tgl).dt.days > 365
        timeliness_score = (1 - is_stale.sum() / n_rows) * 100
    else:
        tgl_sync = pd.to_datetime(df['TGL_SYNC_OSS'], errors='coerce')
        high_sync_lag = (datetime.now() - tgl_sync).dt.days > 30
        timeliness_score = (1 - high_sync_lag.sum() / n_rows) * 100
    results['Ketepatan Waktu'] = timeliness_score

    return results


def generate_professional_scorecard(metrics, title_suffix="CEISA"):
    """Visualisasi Dual-Chart: Hexagon Radar & Horizontal Bar."""
    dim_names = list(metrics.keys())
    dim_scores = list(metrics.values())
    total_dq_score = np.mean(dim_scores)

    TARGET_SCORE = 80
    WARNING_SCORE = 70

    if total_dq_score >= TARGET_SCORE:
        grade, grade_color = 'A (Excellent)', '#4CAF50'
    elif total_dq_score >= WARNING_SCORE:
        grade, grade_color = 'B (Good)', '#8BC34A'
    elif total_dq_score >= 60:
        grade, grade_color = 'C (Fair)', '#FF9800'
    else:
        grade, grade_color = 'D (Poor)', '#F44336'

    fig = plt.figure(figsize=(16, 8))
    fig.suptitle(f'Data Quality Scorecard — {title_suffix} (Before MDM)', fontsize=16, fontweight='bold', y=1.05)

    # ── Chart 1: Radar Chart (Hexagon) ──
    angles = np.linspace(0, 2 * np.pi, len(dim_names), endpoint=False).tolist()
    dim_scores_polar = dim_scores + [dim_scores[0]]
    angles += [angles[0]]

    ax1 = plt.subplot(121, polar=True)
    ax1.set_theta_offset(np.pi / 2)
    ax1.set_theta_direction(-1)

    ax1.plot(angles, dim_scores_polar, 'o-', linewidth=3, color='#2196F3', markersize=8)
    ax1.fill(angles, dim_scores_polar, alpha=0.3, color='#2196F3')
    ax1.plot(angles, [TARGET_SCORE] * len(angles), '--', color='red', alpha=0.6, label=f'Target {TARGET_SCORE}%')

    ax1.set_xticks(angles[:-1])
    ax1.set_xticklabels(dim_names, fontsize=10, fontweight='bold')
    ax1.set_ylim(0, 100)
    ax1.set_yticks([20, 40, 60, 80, 100])
    ax1.set_yticklabels(['20', '40', '60', '80', '100%'], fontsize=8)
    ax1.set_title('DQ Score Dimensions (DMBOK)', fontweight='bold', pad=30)
    ax1.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1), fontsize=9)

    for angle, score in zip(angles[:-1], dim_scores):
        ax1.annotate(f'{score:.1f}%', xy=(angle, score), fontsize=9, ha='center',
                      xytext=(0, 10), textcoords='offset points', color='white',
                      fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', fc='#1565C0', alpha=0.8))

    # ── Chart 2: Horizontal Bar Chart ──
    ax2 = plt.subplot(122)
    y_pos = np.arange(len(dim_names))
    colors2 = [grade_color if s >= TARGET_SCORE else '#FF9800' if s >= WARNING_SCORE else '#F44336' for s in dim_scores]

    bars2 = ax2.barh(y_pos, dim_scores, color=colors2, edgecolor='white', height=0.6)
    ax2.axvline(x=TARGET_SCORE, color='red', linestyle='--', linewidth=2, label=f'Target {TARGET_SCORE}%')

    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(dim_names, fontsize=10, fontweight='bold')
    ax2.set_xlim(0, 105)
    ax2.set_xlabel('Score (%)')
    ax2.set_title(f'DQ Score vs Target\n(Avg Score: {total_dq_score:.1f}/100 | Grade: {grade})', fontweight='bold')

    for bar, score in zip(bars2, dim_scores):
        ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, f'{score:.1f}%', va='center', fontweight='bold', fontsize=10)

    plt.tight_layout()
    filename = f'reports/dq_scorecard_{title_suffix.lower().replace(" ", "_")}.png'
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Chart disimpan: {filename}')


# ───────────────────────── 7. YDATA PROFILING REPORT ─────────────────────────
# Korelasi kategorik (phi_k/cramers) & interaksi numerik dimatikan karena kolom
# identitas (NIB/NPWP/NAMA/ALAMAT) berkardinalitas sangat tinggi (~hampir unik
# per baris) sehingga perhitungan contingency table-nya sangat lambat & tidak
# informatif untuk profiling ini.
YDATA_CORRELATIONS = {
    "auto": {"calculate": False},
    "pearson": {"calculate": True},
    "spearman": {"calculate": False},
    "kendall": {"calculate": False},
    "phi_k": {"calculate": False},
    "cramers": {"calculate": False},
}


def generate_ydata_report(df, name, output_path):
    print(f"\nMembuat ydata_profiling report untuk {name} -> {output_path} ...")
    profile = ProfileReport(
        df,
        title=f'Data Profiling Report — {name} (Before MDM)',
        explorative=True,
        correlations=YDATA_CORRELATIONS,
        interactions={"continuous": False},
    )
    profile.to_file(output_path)
    print(f"Report disimpan: {output_path}")


def build_profiling_index(summary, output_path='reports/profiling_before.html'):
    """Halaman index ringkas yang menggabungkan hasil profiling OSS & CEISA
    (overview, missing value, duplikat, format validation, baseline DQ score)
    serta link ke laporan ydata_profiling lengkap masing-masing dataset."""

    def fmt_checks(checks):
        rows = ''.join(
            f"<tr><td>{label}</td><td>{valid:,}</td><td>{total:,}</td><td>{valid / total * 100 if total else 0:.2f}%</td></tr>"
            for label, (valid, total) in checks.items()
        )
        return f"<table><tr><th>Check</th><th>Valid</th><th>Total</th><th>%</th></tr>{rows}</table>"

    def fmt_missing(table):
        if table.empty:
            return "<p>Tidak ada missing value.</p>"
        rows = ''.join(
            f"<tr><td>{idx}</td><td>{row.n_missing:,}</td><td>{row.pct_missing:.2f}%</td><td>{row.severity}</td></tr>"
            for idx, row in table.iterrows()
        )
        return f"<table><tr><th>Kolom</th><th>Jumlah Missing</th><th>% Missing</th><th>Severity</th></tr>{rows}</table>"

    def fmt_dq(metrics):
        rows = ''.join(f"<tr><td>{k}</td><td>{v:.2f}%</td></tr>" for k, v in metrics.items())
        avg = np.mean(list(metrics.values()))
        return f"<table><tr><th>Dimensi DMBOK</th><th>Score</th></tr>{rows}<tr><th>Overall</th><th>{avg:.2f}%</th></tr></table>"

    html = f"""<!DOCTYPE html>
<html lang="id">
<head>
<meta charset="utf-8">
<title>Data Profiling Report - Before MDM (Tahap 1)</title>
<style>
  body {{ font-family: Arial, Helvetica, sans-serif; margin: 40px; color: #222; }}
  h1 {{ color: #1565C0; }}
  h2 {{ border-bottom: 2px solid #1565C0; padding-bottom: 4px; margin-top: 40px; }}
  table {{ border-collapse: collapse; margin: 10px 0 20px 0; width: 100%; max-width: 800px; }}
  th, td {{ border: 1px solid #ccc; padding: 6px 10px; text-align: left; font-size: 14px; }}
  th {{ background: #1565C0; color: white; }}
  tr:nth-child(even) {{ background: #f5f5f5; }}
  .summary {{ background: #E3F2FD; padding: 12px 16px; border-radius: 6px; }}
  a.btn {{ display: inline-block; margin: 8px 12px 8px 0; padding: 8px 16px; background: #1565C0;
           color: white; text-decoration: none; border-radius: 4px; }}
</style>
</head>
<body>
<h1>Data Profiling Report — Before MDM (Tahap 1)</h1>
<p class="summary">
  Mini Project Kelompok 5 (DJBC) — Master Data Importir &amp; Eksportir Nasional.<br>
  Profiling dilakukan terhadap data <b>OSS</b> ({summary['n_oss']:,} baris) dan
  <b>CEISA</b> ({summary['n_ceisa']:,} baris) sebelum proses cleansing &amp; Golden Record.
</p>

<h2>Laporan ydata_profiling Lengkap</h2>
<a class="btn" href="profiling_before_oss.html">OSS — Full Profiling Report</a>
<a class="btn" href="profiling_before_ceisa.html">CEISA — Full Profiling Report</a>

<h2>1. Dataset Overview</h2>
<table>
<tr><th>Dataset</th><th>Jumlah Baris</th><th>Jumlah Kolom</th></tr>
<tr><td>OSS</td><td>{summary['n_oss']:,}</td><td>{summary['n_cols_oss']}</td></tr>
<tr><td>CEISA</td><td>{summary['n_ceisa']:,}</td><td>{summary['n_cols_ceisa']}</td></tr>
</table>

<h2>2. Missing Value Analysis</h2>
<h3>OSS</h3>
{fmt_missing(summary['missing_oss'])}
<h3>CEISA</h3>
{fmt_missing(summary['missing_ceisa'])}

<h2>3. Duplicate Analysis</h2>
<table>
<tr><th>Metrik</th><th>OSS</th><th>CEISA</th></tr>
<tr><td>Full duplicate (baris identik)</td><td>{summary['dup_oss']['full_duplicate']:,}</td><td>{summary['dup_ceisa']['full_duplicate']:,}</td></tr>
<tr><td>Duplicate by NIB</td><td>{summary['dup_oss']['duplicate_by_nib']:,}</td><td>{summary['dup_ceisa']['duplicate_by_nib']:,}</td></tr>
</table>
<p><i>Catatan: Duplicate by NIB di OSS adalah anomali (NIB harus unik), sedangkan di CEISA sebagian besar
adalah Data Mart Snapshot Duplicate (expected) — lihat <code>docs/02_business_rules.md</code> Section 5.</i></p>

<h2>4. Format Validation</h2>
<h3>OSS</h3>
{fmt_checks(summary['fmt_oss'])}
<h3>CEISA</h3>
{fmt_checks(summary['fmt_ceisa'])}

<h2>5. Baseline Data Quality Score (4 Dimensi DMBOK)</h2>
<h3>OSS</h3>
{fmt_dq(summary['dq_oss'])}
<h3>CEISA</h3>
{fmt_dq(summary['dq_ceisa'])}

<p><i>Generated by Source/step05_profiling.py</i></p>
</body>
</html>
"""
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html)
    print(f"\nProfiling index disimpan: {output_path}")




### Eksekusi Profiling - OSS

Menjalankan profiling untuk dataset OSS.

In [ ]:
# Load data OSS
df_oss = pd.read_csv(f'{DATA_RAW}/oss_nib_data.csv', dtype={'NIB': str, 'NPWP_PERSEROAN': str})
print(f'Dataset OSS: {len(df_oss):,} baris')

dataset_overview(df_oss, 'OSS')
descriptive_stats(df_oss, 'OSS')
missing_oss = missing_value_analysis(df_oss, 'OSS')
dup_oss = duplicate_analysis(df_oss, 'OSS', 'NAMA_PERSEROAN')
fmt_oss = format_validation(df_oss, 'OSS')

In [ ]:
# Profiling CEISA
df_ceisa = pd.read_csv(f'{DATA_RAW}/ceisa_data.csv', dtype={'NIB': str, 'NPWP': str})
print(f'Dataset CEISA: {len(df_ceisa):,} baris')

dataset_overview(df_ceisa, 'CEISA')
descriptive_stats(df_ceisa, 'CEISA')
missing_ceisa = missing_value_analysis(df_ceisa, 'CEISA')
dup_ceisa = duplicate_analysis(df_ceisa, 'CEISA', 'NAMA_PERUSAHAAN')
fmt_ceisa = format_validation(df_ceisa, 'CEISA')

### Baseline Data Quality Score

Menghitung skor kualitas data 4 dimensi DMBOK sebagai baseline.

In [ ]:
# Baseline DQ Score
oss_metrics = calculate_dq_metrics(df_oss, 'OSS')
ceisa_metrics = calculate_dq_metrics(df_ceisa, 'CEISA')

dq_compare = pd.DataFrame({
    'Dimensi': list(oss_metrics.keys()),
    'OSS (%)': [f'{oss_metrics[k]:.2f}%' for k in oss_metrics],
    'CEISA (%)': [f'{ceisa_metrics[k]:.2f}%' for k in ceisa_metrics],
})
display(dq_compare)

avg_oss = sum(oss_metrics.values()) / len(oss_metrics)
avg_ceisa = sum(ceisa_metrics.values()) / len(ceisa_metrics)
print(f'\nOverall DQ Score:')
print(f'  OSS   : {avg_oss:.2f}%')
print(f'  CEISA : {avg_ceisa:.2f}%')

# Visualisasi radar chart
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
dims = list(oss_metrics.keys())
angles = np.linspace(0, 2*np.pi, len(dims), endpoint=False).tolist() + [0]
oss_vals = list(oss_metrics.values()) + [list(oss_metrics.values())[0]]
ceisa_vals = list(ceisa_metrics.values()) + [list(ceisa_metrics.values())[0]]

ax.plot(angles, oss_vals, 'o-', lw=2, color='#1565C0', label='OSS')
ax.fill(angles, oss_vals, alpha=0.1, color='#1565C0')
ax.plot(angles, ceisa_vals, 'o-', lw=2, color='#F44336', label='CEISA')
ax.fill(angles, ceisa_vals, alpha=0.1, color='#F44336')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(dims, fontsize=11, fontweight='bold')
ax.set_ylim(0, 100)
ax.set_title('Baseline DQ Score - OSS vs CEISA', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.2, 1.1))
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/dq_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

### Laporan YData Profiling

Menghasilkan laporan profiling interaktif untuk OSS dan CEISA.

In [ ]:
# YData Profiling Reports
print('Membuat YData Profiling untuk OSS...')
ProfileReport(
    df_oss,
    title='Data Profiling Report - OSS (Before MDM)',
    explorative=True,
    correlations={"auto": {"calculate": False}, "pearson": {"calculate": True},
                  "spearman": {"calculate": False}, "kendall": {"calculate": False},
                  "phi_k": {"calculate": False}, "cramers": {"calculate": False}},
    interactions={"continuous": False}
).to_file(f'{REPORTS_DIR}/profiling_before_oss.html')
print(f'  Report OSS: {REPORTS_DIR}/profiling_before_oss.html')

print('\nMembuat YData Profiling untuk CEISA...')
ProfileReport(
    df_ceisa,
    title='Data Profiling Report - CEISA (Before MDM)',
    explorative=True,
    correlations={"auto": {"calculate": False}, "pearson": {"calculate": True},
                  "spearman": {"calculate": False}, "kendall": {"calculate": False},
                  "phi_k": {"calculate": False}, "cramers": {"calculate": False}},
    interactions={"continuous": False}
).to_file(f'{REPORTS_DIR}/profiling_before_ceisa.html')
print(f'  Report CEISA: {REPORTS_DIR}/profiling_before_ceisa.html')
print('\nTahap 1 - Data Profiling selesai!')

# TAHAP 2: DATA CLEANSING & STANDARDIZATION

**Tujuan:** Membersihkan dan menstandarisasi data OSS & CEISA agar konsisten.

### Penjelasan

Kegiatan:
1. AuditTrail - catat setiap operasi cleansing
2. Standardisasi NAMA - uppercase, normalisasi prefix
3. Standardisasi ALAMAT - title case, normalisasi singkatan
4. Standardisasi identifier - NIB, NPWP, KODE_POS, Telepon
5. CEISA Data Mart Dedup - 1 baris per NIB (snapshot terbaru)
6. Missing value handling - flag field wajib
7. Validasi referensial - kode wilayah
8. Quality Gate - pemeriksaan otomatis

In [ ]:
# [TAHAP 2] DATA CLEANSING & STANDARDIZATION
# Berdasarkan: docs/tahapan/tahap2_cleansing_standardization.md & docs/02_business_rules.md
#
# Pipeline:
#   1. AuditTrail - catat setiap operasi cleansing (apa, berapa baris terdampak)
#   2. Standardisasi NAMA (uppercase, normalisasi prefix PT/CV/UD/Firma/Perum)
#   3. Standardisasi ALAMAT (title case + normalisasi singkatan Jl./No./RT/RW/Kec./Kel.)
#   4. Standardisasi identifier: NIB (13 digit), NPWP (XX.XXX.XXX.X-XXX.XXX),
#      KODE_POS (5 digit), NOMOR_TELPON CEISA (+62...)
#   5. CEISA Data Mart Dedup (1 baris per NIB, snapshot TGL_SYNC_OSS terbaru)
#   6. Missing value handling (wajib: flag saja, opsional: dibiarkan null)
#   7. Validasi referensial PERSEROAN_DAERAH_ID/DAERAH_ID terhadap PROVINSI
#   8. Quality Gate sebelum export
#   9. Export oss_cleaned.csv, ceisa_cleaned.csv, dataset_clean.csv, audit_trail.csv
#  10. Perbandingan skor DQ before vs after (4 dimensi DMBOK)

import pandas as pd
import re
from datetime import datetime

# PROVINSI already defined above
# from Source.step05_profiling import (
    NIB_PATTERN, NPWP_PATTERN, calculate_dq_metrics,
)

DUMMY_NIB = '0' * 13  # NIB dummy/kosong - sama dengan Source/step07_matching.py

MAND_COLS_OSS = ['NIB', 'NPWP_PERSEROAN', 'NAMA_PERSEROAN', 'STATUS_NIB']
MAND_COLS_CEISA = ['NIB', 'NPWP', 'NAMA_PERUSAHAAN', 'STATUS_NIB']


# ───────────────────────── 1. AUDIT TRAIL ─────────────────────────
class AuditTrail:
    def __init__(self):
        self.entries = []

    def log(self, dataset, operation, field, n_affected, total, description):
        pct = round(n_affected / total * 100, 2) if total else 0.0
        self.entries.append({
            'dataset': dataset,
            'timestamp': datetime.now().isoformat(),
            'operation': operation,
            'field': field,
            'n_affected': int(n_affected),
            'pct_affected': pct,
            'description': description,
        })

    def to_dataframe(self):
        return pd.DataFrame(self.entries)


# ───────────────────────── 2. STANDARDISASI NAMA ─────────────────────────
def standardize_nama(nama):
    """Uppercase, rapikan spasi, normalisasi prefix PT/CV/UD/Firma/Perum."""
    if pd.isna(nama):
        return nama
    s = re.sub(r'\s+', ' ', str(nama).strip().upper())
    prefix_map = {
        r'^P\.?\s*T\.?\s+': 'PT ',
        r'^C\.?\s*V\.?\s+': 'CV ',
        r'^U\.?\s*D\.?\s+': 'UD ',
        r'^FIRMA\.?\s+': 'FIRMA ',
        r'^PERUM\.?\s+': 'PERUM ',
    }
    for pattern, repl in prefix_map.items():
        s = re.sub(pattern, repl, s)
    return s


# ───────────────────────── 3. STANDARDISASI ALAMAT ─────────────────────────
_ALAMAT_ABBR = {
    r'\bJl\b\.?': 'Jl.',
    r'\bGg\b\.?': 'Gg.',
    r'\bNo\b\.?': 'No.',
    r'\bRt\b\.?': 'RT',
    r'\bRw\b\.?': 'RW',
    r'\bKec\b\.?': 'Kec.',
    r'\bKel\b\.?': 'Kel.',
    r'\bDr\b\.?': 'Dr.',
}


def standardize_alamat(alamat):
    """Title case + normalisasi singkatan (Jl., No., RT/RW, Kec., Kel.)."""
    if pd.isna(alamat):
        return alamat
    s = re.sub(r'\s+', ' ', str(alamat).strip()).title()
    for pattern, repl in _ALAMAT_ABBR.items():
        s = re.sub(pattern, repl, s)
    return s


# ───────────────────────── 4. STANDARDISASI IDENTIFIER ─────────────────────────
def standardize_nib(nib):
    """Hanya digit, pad/trim ke 13 digit."""
    if pd.isna(nib):
        return nib
    digits = re.sub(r'\D', '', str(nib))
    if len(digits) > 13:
        digits = digits[:13]
    elif len(digits) < 13:
        digits = digits.zfill(13)
    return digits


def standardize_npwp(npwp):
    """Format ke XX.XXX.XXX.X-XXX.XXX jika 15 digit; selain itu tetap 'kotor'
    (akan ditandai IS_NPWP_INVALID, sesuai anomali 'Inconsistent NPWP')."""
    if pd.isna(npwp):
        return npwp
    digits = re.sub(r'\D', '', str(npwp))
    if len(digits) == 15:
        return f"{digits[0:2]}.{digits[2:5]}.{digits[5:8]}.{digits[8]}-{digits[9:12]}.{digits[12:15]}"
    return digits


def standardize_kode_pos(kode_pos):
    """Pastikan KODE_POS 5 digit; selain itu dikosongkan (field opsional).
    Catatan: KODE_POS terbaca float64 sehingga leading zero (mis. '08123'
    -> 8123.0) sudah hilang sejak data mentah - nilai seperti ini tidak
    bisa dipastikan 5 digit sehingga ikut dikosongkan."""
    if pd.isna(kode_pos):
        return None
    val = kode_pos
    if isinstance(val, float) and val.is_integer():
        val = int(val)
    digits = re.sub(r'\D', '', str(val))
    if len(digits) == 5:
        return int(digits)
    return None


def standardize_telpon(telp):
    """Normalisasi nomor telepon CEISA ke format +62... (hapus separator,
    kode negara/awalan 0 ganda, dan leading zero pada nomor lokal)."""
    if pd.isna(telp):
        return telp
    digits = re.sub(r'\D', '', str(telp))
    if digits.startswith('62'):
        digits = digits[2:]
    digits = digits.lstrip('0')
    return '+62' + digits


def _kode_pos_changed(before_series, after_series):
    """Hitung baris yang berubah (dikosongkan / nilai berbeda) akibat
    standardisasi KODE_POS."""
    n_changed = 0
    for b, a in zip(before_series, after_series):
        b_blank, a_blank = pd.isna(b), pd.isna(a)
        if b_blank != a_blank:
            n_changed += 1
        elif not b_blank and not a_blank and int(b) != int(a):
            n_changed += 1
    return n_changed


# ───────────────────────── 5. CEISA DATA MART DEDUP ─────────────────────────
def dedup_ceisa(df, audit):
    """1 baris per NIB, sisakan snapshot dengan TGL_SYNC_OSS terbaru.
    NIB dummy ('0'*13) merepresentasikan banyak entitas berbeda yang belum
    punya NIB valid (lihat Source/step07_matching.py: DUMMY_NIB dikecualikan
    dari exact-match), sehingga untuk baris dummy dedup dilakukan per
    ID_PERUSAHAAN (bukan per NIB) agar snapshot duplikat tetap tereliminasi
    tanpa menghapus entitas berbeda."""
    n_before = len(df)
    df = df.copy()
    df['_TGL_SYNC_SORT'] = pd.to_datetime(df['TGL_SYNC_OSS'], errors='coerce')

    is_dummy = df['NIB'] == DUMMY_NIB
    df_real = df[~is_dummy].sort_values('_TGL_SYNC_SORT', ascending=False)
    df_dummy = df[is_dummy].sort_values('_TGL_SYNC_SORT', ascending=False)

    n_multi_nib = (df_real['NIB'].value_counts() > 1).sum()
    df_real_dedup = df_real.drop_duplicates(subset='NIB', keep='first')

    n_multi_id_dummy = (df_dummy['ID_PERUSAHAAN'].value_counts() > 1).sum()
    df_dummy_dedup = df_dummy.drop_duplicates(subset='ID_PERUSAHAAN', keep='first')

    result = pd.concat([df_real_dedup, df_dummy_dedup], ignore_index=True).drop(columns='_TGL_SYNC_SORT')
    n_removed = n_before - len(result)

    audit.log('CEISA', 'DEDUP_SNAPSHOT', 'NIB', n_removed, n_before,
              f"{n_multi_nib} NIB (non-dummy) punya >1 snapshot data mart - disisakan baris "
              f"TGL_SYNC_OSS terbaru; {n_multi_id_dummy} ID_PERUSAHAAN ber-NIB dummy "
              f"({DUMMY_NIB}) juga di-dedup agar ID_PERUSAHAAN tetap unik")
    return result


# ───────────────────────── 8. QUALITY GATE ─────────────────────────
def quality_gate(df, dataset_name, mand_cols):
    print(f"\n--- Quality Gate: {dataset_name} ---")
    issues = []

    nib_invalid = (~df['NIB'].astype(str).str.match(NIB_PATTERN)).sum()
    if nib_invalid:
        issues.append(f"{nib_invalid} NIB tidak 13 digit setelah standardisasi")

    n_incomplete = df[mand_cols].isnull().any(axis=1).sum()
    if n_incomplete:
        issues.append(f"{n_incomplete} baris field wajib masih kosong")

    if dataset_name == 'CEISA':
        non_dummy = df[df['NIB'] != DUMMY_NIB]
        n_dup = non_dummy['NIB'].duplicated().sum()
        if n_dup:
            issues.append(f"{n_dup} NIB (non-dummy) masih duplikat setelah dedup")

    if issues:
        print("STATUS: FAIL")
        for issue in issues:
            print(f"  - {issue}")
    else:
        print("STATUS: PASS - tidak ada isu kritis terdeteksi")

    return len(issues) == 0


# ───────────────────────── 10. PERBANDINGAN DQ SCORE ─────────────────────────
def print_dq_comparison(metrics_before, metrics_after, name):
    print(f"\n--- Perbandingan Skor DQ (4 Dimensi DMBOK) — {name} ---")
    print(f"  {'Dimensi':<20}{'Before':>10}{'After':>10}{'Delta':>10}")
    for dim in metrics_before:
        before, after = metrics_before[dim], metrics_after[dim]
        print(f"  {dim:<20}{before:>9.2f}%{after:>9.2f}%{after - before:>+9.2f}%")
    avg_before = sum(metrics_before.values()) / len(metrics_before)
    avg_after = sum(metrics_after.values()) / len(metrics_after)
    print(f"  {'TOTAL DQ SCORE':<20}{avg_before:>9.2f}%{avg_after:>9.2f}%{avg_after - avg_before:>+9.2f}%")


# ───────────────────────── PIPELINE PER DATASET ─────────────────────────
def clean_oss(df_raw, audit):
    df = df_raw.copy()
    n = len(df)

    before = df['NAMA_PERSEROAN'].copy()
    df['NAMA_PERSEROAN'] = df['NAMA_PERSEROAN'].apply(standardize_nama)
    audit.log('OSS', 'STANDARDIZE_NAME', 'NAMA_PERSEROAN', (df['NAMA_PERSEROAN'] != before).sum(), n,
              "Uppercase, rapikan spasi, normalisasi prefix PT/CV/Firma/Perum/UD")

    before = df['ALAMAT_PERSEROAN'].copy()
    df['ALAMAT_PERSEROAN'] = df['ALAMAT_PERSEROAN'].apply(standardize_alamat)
    audit.log('OSS', 'STANDARDIZE_ADDRESS', 'ALAMAT_PERSEROAN', (df['ALAMAT_PERSEROAN'] != before).sum(), n,
              "Title case, normalisasi singkatan (Jl., No., RT/RW, Kec., Kel.)")

    before = df['NIB'].copy()
    df['NIB'] = df['NIB'].apply(standardize_nib)
    audit.log('OSS', 'STANDARDIZE_NIB', 'NIB', (df['NIB'] != before).sum(), n,
              "Hanya digit, pad/trim ke 13 digit")

    before = df['NPWP_PERSEROAN'].copy()
    df['NPWP_PERSEROAN'] = df['NPWP_PERSEROAN'].apply(standardize_npwp)
    audit.log('OSS', 'STANDARDIZE_NPWP', 'NPWP_PERSEROAN', (df['NPWP_PERSEROAN'] != before).sum(), n,
              "Format ke XX.XXX.XXX.X-XXX.XXX (jika 15 digit)")

    df['IS_NPWP_INVALID'] = ~df['NPWP_PERSEROAN'].astype(str).str.match(NPWP_PATTERN)
    audit.log('OSS', 'FLAG_VALIDITY', 'NPWP_PERSEROAN', df['IS_NPWP_INVALID'].sum(), n,
              "Format NPWP_PERSEROAN tidak sesuai XX.XXX.XXX.X-XXX.XXX - ditandai IS_NPWP_INVALID")

    before = df['KODE_POS_PERSEROAN'].copy()
    df['KODE_POS_PERSEROAN'] = df['KODE_POS_PERSEROAN'].apply(standardize_kode_pos)
    audit.log('OSS', 'STANDARDIZE_KODEPOS', 'KODE_POS_PERSEROAN', _kode_pos_changed(before, df['KODE_POS_PERSEROAN']), n,
              "Pastikan 5 digit, selain itu dikosongkan (field opsional)")

    df['IS_INCOMPLETE'] = df[MAND_COLS_OSS].isnull().any(axis=1)
    audit.log('OSS', 'FLAG_INCOMPLETE', '+'.join(MAND_COLS_OSS), df['IS_INCOMPLETE'].sum(), n,
              "Field wajib kosong - ditandai untuk review, tidak diisi paksa")

    daerah_str = df['PERSEROAN_DAERAH_ID'].apply(lambda x: f'{int(x):02d}' if pd.notna(x) else None)
    df['IS_DAERAH_VALID'] = daerah_str.isin(PROVINSI.keys())
    audit.log('OSS', 'VALIDATE_REF', 'PERSEROAN_DAERAH_ID', (~df['IS_DAERAH_VALID']).sum(), n,
              "Kode wilayah tidak ditemukan di 34 kode referensi PROVINSI")

    tgl = pd.to_datetime(df['TGL_PERUBAHAN_NIB'], errors='coerce')
    df['IS_STALE'] = (datetime.now() - tgl).dt.days > 365
    audit.log('OSS', 'FLAG_TIMELINESS', 'TGL_PERUBAHAN_NIB', df['IS_STALE'].sum(), n,
              "IS_STALE = True jika TGL_PERUBAHAN_NIB > 1 tahun dari sekarang")

    df['IS_CLEANED'] = True
    df['CLEANSING_TIMESTAMP'] = datetime.now().isoformat()
    return df


def clean_ceisa(df_raw, audit):
    df = df_raw.copy()
    n = len(df)

    before = df['NAMA_PERUSAHAAN'].copy()
    df['NAMA_PERUSAHAAN'] = df['NAMA_PERUSAHAAN'].apply(standardize_nama)
    audit.log('CEISA', 'STANDARDIZE_NAME', 'NAMA_PERUSAHAAN', (df['NAMA_PERUSAHAAN'] != before).sum(), n,
              "Uppercase, rapikan spasi, normalisasi prefix PT/CV/Firma/Perum/UD")

    before = df['ALAMAT_PERUSAHAAN'].copy()
    df['ALAMAT_PERUSAHAAN'] = df['ALAMAT_PERUSAHAAN'].apply(standardize_alamat)
    audit.log('CEISA', 'STANDARDIZE_ADDRESS', 'ALAMAT_PERUSAHAAN', (df['ALAMAT_PERUSAHAAN'] != before).sum(), n,
              "Title case, normalisasi singkatan (Jl., No., RT/RW, Kec., Kel.)")

    before = df['NIB'].copy()
    df['NIB'] = df['NIB'].apply(standardize_nib)
    audit.log('CEISA', 'STANDARDIZE_NIB', 'NIB', (df['NIB'] != before).sum(), n,
              "Hanya digit, pad/trim ke 13 digit")

    before = df['NPWP'].copy()
    df['NPWP'] = df['NPWP'].apply(standardize_npwp)
    n_npwp_changed = (df['NPWP'] != before).sum()
    n_still_dirty = (~df['NPWP'].astype(str).str.match(NPWP_PATTERN)).sum()
    audit.log('CEISA', 'STANDARDIZE_NPWP', 'NPWP', n_npwp_changed, n,
              f"Format ke XX.XXX.XXX.X-XXX.XXX (jika 15 digit); {n_still_dirty} masih tidak baku "
              f"(anomali \"Inconsistent NPWP\" dipertahankan sesuai docs/02_business_rules.md §1)")

    df['IS_NPWP_INVALID'] = ~df['NPWP'].astype(str).str.match(NPWP_PATTERN)
    audit.log('CEISA', 'FLAG_VALIDITY', 'NPWP', df['IS_NPWP_INVALID'].sum(), n,
              "Format NPWP tidak sesuai XX.XXX.XXX.X-XXX.XXX - ditandai IS_NPWP_INVALID")

    before = df['KODE_POS'].copy()
    df['KODE_POS'] = df['KODE_POS'].apply(standardize_kode_pos)
    audit.log('CEISA', 'STANDARDIZE_KODEPOS', 'KODE_POS', _kode_pos_changed(before, df['KODE_POS']), n,
              "Pastikan 5 digit, selain itu dikosongkan (field opsional)")

    before = df['NOMOR_TELPON'].copy()
    df['NOMOR_TELPON'] = df['NOMOR_TELPON'].apply(standardize_telpon)
    audit.log('CEISA', 'STANDARDIZE_PHONE', 'NOMOR_TELPON', (df['NOMOR_TELPON'] != before).sum(), n,
              "Normalisasi ke format +62...")

    df = dedup_ceisa(df, audit)
    n = len(df)

    df['IS_INCOMPLETE'] = df[MAND_COLS_CEISA].isnull().any(axis=1)
    audit.log('CEISA', 'FLAG_INCOMPLETE', '+'.join(MAND_COLS_CEISA), df['IS_INCOMPLETE'].sum(), n,
              "Field wajib kosong - ditandai untuk review, tidak diisi paksa")

    daerah_str = df['DAERAH_ID'].apply(lambda x: f'{int(x):02d}' if pd.notna(x) else None)
    df['IS_DAERAH_VALID'] = daerah_str.isin(PROVINSI.keys())
    audit.log('CEISA', 'VALIDATE_REF', 'DAERAH_ID', (~df['IS_DAERAH_VALID']).sum(), n,
              "Kode wilayah tidak ditemukan di 34 kode referensi PROVINSI")

    tgl_sync = pd.to_datetime(df['TGL_SYNC_OSS'], errors='coerce')
    df['HIGH_SYNC_LAG'] = (datetime.now() - tgl_sync).dt.days > 30
    audit.log('CEISA', 'FLAG_TIMELINESS', 'TGL_SYNC_OSS', df['HIGH_SYNC_LAG'].sum(), n,
              "HIGH_SYNC_LAG = True jika TGL_SYNC_OSS > 30 hari dari sekarang")

    df['IS_CLEANED'] = True
    df['CLEANSING_TIMESTAMP'] = datetime.now().isoformat()
    return df




# TAHAP 3: DUPLICATE DETECTION & MATCHING

**Tujuan:** Mencocokkan data OSS dan CEISA untuk mengidentifikasi entitas yang sama.

### Penjelasan

Prioritas matching:
1. Exact match NIB (Prioritas 1)
2. Exact match NPWP (Prioritas 2)
3. Fuzzy match nama + alamat (Prioritas 3)
4. Composite similarity score (NPWP 20% + Nama 50% + Alamat 30%)
5. Duplicate clustering internal

In [ ]:
# [TAHAP 3] DUPLICATE DETECTION & MATCHING
# Berdasarkan: docs/tahapan/tahap3_duplicate_matching.md & docs/02_business_rules.md §2
#
# Matching utama dilakukan ANTAR DUA DATASET (OSS vs CEISA), bukan dalam satu dataset:
#   1. Exact match NIB      (Prioritas 1)
#   2. Exact match NPWP     (Prioritas 2, untuk sisa yang belum match -> tangkap "NIB Typo")
#   3. Fuzzy match nama/alamat (Prioritas 3, untuk sisa yang belum match)
#   4. Duplicate clustering internal per sumber (OSS: Duplicate Entry by NIB,
#      CEISA: nama/alamat sangat mirip dengan NIB berbeda)

import pandas as pd
import numpy as np
import re
import recordlinkage
from fuzzywuzzy import fuzz
import jellyfish
import networkx as nx
import matplotlib.pyplot as plt

# ============================================================
# KONFIGURASI THRESHOLD & BOBOT (Langkah 6-7)
# ============================================================

UPPER_THRESHOLD = 0.85   # composite_score >= ini      -> FUZZY_MATCH (kandidat valid)
LOWER_THRESHOLD = 0.70   # composite_score <  ini       -> NON_MATCH (dibuang)
                          # di antara keduanya           -> FUZZY_REVIEW (perlu review manual)

# Bobot composite score: NPWP (jika sebagian mirip) + Nama + Alamat, total = 1.0
WEIGHTS = {'npwp': 0.20, 'nama': 0.50, 'alamat': 0.30}

DUMMY_NIB = '0' * 13  # NIB dummy/kosong hasil generate_nib_invalid() - dikecualikan dari exact match


# ============================================================
# LANGKAH 2: PERSIAPAN MATCHING KEYS
# ============================================================

def normalize_for_matching(text):
    """Normalisasi agresif untuk matching: uppercase, normalisasi prefix badan usaha, hapus simbol."""
    if pd.isna(text):
        return ''
    s = str(text).upper()
    for prefix in ['PT', 'CV', 'UD', 'FIRMA', 'PERUM']:
        s = re.sub(rf'\b{prefix}\.?\b', prefix, s)
    s = re.sub(r'[^A-Z0-9\s]', '', s)
    s = re.sub(r'\s+', ' ', s)
    return s.strip()


def extract_digits(value):
    """Ekstrak hanya digit dari sebuah nilai (NIB/NPWP) - tanpa peduli separator."""
    if pd.isna(value):
        return ''
    return re.sub(r'\D', '', str(value))


def prepare_matching_keys(df_oss, df_ceisa):
    """Tambahkan kolom nib_digits, npwp_digits, nama_key, alamat_key, region_key."""
    df_oss = df_oss.copy()
    df_ceisa = df_ceisa.copy()

    df_oss['nib_digits'] = df_oss['NIB'].apply(extract_digits)
    df_oss['npwp_digits'] = df_oss['NPWP_PERSEROAN'].apply(extract_digits)
    df_oss['nama_key'] = df_oss['NAMA_PERSEROAN'].apply(normalize_for_matching)
    df_oss['alamat_key'] = df_oss['ALAMAT_PERSEROAN'].apply(normalize_for_matching)
    df_oss['region_key'] = df_oss['PERSEROAN_DAERAH_ID'].astype(str)

    df_ceisa['nib_digits'] = df_ceisa['NIB'].apply(extract_digits)
    df_ceisa['npwp_digits'] = df_ceisa['NPWP'].apply(extract_digits)
    df_ceisa['nama_key'] = df_ceisa['NAMA_PERUSAHAAN'].apply(normalize_for_matching)
    df_ceisa['alamat_key'] = df_ceisa['ALAMAT_PERUSAHAAN'].apply(normalize_for_matching)
    df_ceisa['region_key'] = df_ceisa['DAERAH_ID'].astype(str)

    return df_oss, df_ceisa


# ============================================================
# LANGKAH 3-4: EXACT MATCHING (NIB -> NPWP)
# ============================================================

def exact_match_nib(df_oss, df_ceisa):
    """Prioritas 1: join OSS-CEISA pada nib_digits (kecuali NIB dummy '0000000000000')."""
    left = df_oss[df_oss['nib_digits'] != DUMMY_NIB]
    right = df_ceisa[df_ceisa['nib_digits'] != DUMMY_NIB]

    merged = left.merge(right, on='nib_digits', suffixes=('_OSS', '_CEISA'))
    merged = merged[merged['nib_digits'] != '']

    return pd.DataFrame({
        'NIB_OSS': merged['NIB_OSS'],
        'ID_PERUSAHAAN_CEISA': merged['ID_PERUSAHAAN'],
        'NIB_CEISA': merged['NIB_CEISA'],
        'match_type': 'EXACT_NIB',
        'similarity_score': 1.0,
    })


def exact_match_npwp(df_oss, df_ceisa):
    """Prioritas 2: untuk sisa yang belum match NIB, join pada npwp_digits.
    Menangkap kasus anomali 'NIB Typo' - NIB beda tapi NPWP sama."""
    merged = df_oss.merge(df_ceisa, on='npwp_digits', suffixes=('_OSS', '_CEISA'))
    merged = merged[merged['npwp_digits'] != '']

    return pd.DataFrame({
        'NIB_OSS': merged['NIB_OSS'],
        'ID_PERUSAHAAN_CEISA': merged['ID_PERUSAHAAN'],
        'NIB_CEISA': merged['NIB_CEISA'],
        'match_type': 'EXACT_NPWP',
        'similarity_score': 1.0,
    })


# ============================================================
# LANGKAH 5-6: FUZZY MATCHING & COMPOSITE SCORE
# ============================================================

def fuzzy_match(remaining_oss, remaining_ceisa):
    """Prioritas 3: blocking per region_key (recordlinkage), lalu hitung composite
    similarity score (NPWP + Nama + Alamat) untuk setiap kandidat pasang."""
    cols = ['NIB_OSS', 'ID_PERUSAHAAN_CEISA', 'NIB_CEISA', 'nama_oss', 'nama_ceisa',
            'sim_npwp', 'sim_nama', 'sim_alamat', 'composite_score']

    if remaining_oss.empty or remaining_ceisa.empty:
        return pd.DataFrame(columns=cols)

    oss_idx = remaining_oss.reset_index(drop=True)
    ceisa_idx = remaining_ceisa.set_index('ID_PERUSAHAAN')

    indexer = recordlinkage.Index()
    indexer.block(left_on='region_key', right_on='region_key')
    candidate_links = indexer.index(oss_idx, ceisa_idx)

    results = []
    for pos, id_perusahaan in candidate_links:
        r1 = oss_idx.loc[pos]
        r2 = ceisa_idx.loc[id_perusahaan]

        # Similarity nama: Jaro-Winkler vs Token Set Ratio, ambil yang terbaik
        sim_nama = max(
            jellyfish.jaro_winkler_similarity(r1['nama_key'], r2['nama_key']),
            fuzz.token_set_ratio(r1['nama_key'], r2['nama_key']) / 100,
        )

        # Similarity alamat: Token Set Ratio
        sim_alamat = fuzz.token_set_ratio(r1['alamat_key'], r2['alamat_key']) / 100

        # Similarity NPWP: 1.0 jika digit identik, partial ratio jika sebagian mirip
        d1, d2 = r1['npwp_digits'], r2['npwp_digits']
        if d1 and d2:
            sim_npwp = 1.0 if d1 == d2 else fuzz.ratio(d1, d2) / 100
        else:
            sim_npwp = 0.0

        composite = (
            sim_npwp * WEIGHTS['npwp']
            + sim_nama * WEIGHTS['nama']
            + sim_alamat * WEIGHTS['alamat']
        )

        results.append({
            'NIB_OSS': r1['NIB'],
            'ID_PERUSAHAAN_CEISA': id_perusahaan,
            'NIB_CEISA': r2['NIB'],
            'nama_oss': r1['NAMA_PERSEROAN'],
            'nama_ceisa': r2['NAMA_PERUSAHAAN'],
            'sim_npwp': round(sim_npwp, 4),
            'sim_nama': round(sim_nama, 4),
            'sim_alamat': round(sim_alamat, 4),
            'composite_score': round(composite, 4),
        })

    return pd.DataFrame(results, columns=cols)


def classify_pair(score):
    if score >= UPPER_THRESHOLD:
        return 'FUZZY_MATCH'
    elif score >= LOWER_THRESHOLD:
        return 'FUZZY_REVIEW'
    else:
        return 'NON_MATCH'


# ============================================================
# LANGKAH 7: VISUALISASI DISTRIBUSI & THRESHOLD ANALYSIS
# ============================================================

def plot_score_distribution(df_fuzzy, output_path=f'{REPORTS_DIR}/composite_score_distribution.png'):
    if df_fuzzy.empty:
        print('   (tidak ada kandidat fuzzy untuk divisualisasikan)')
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Distribusi Composite Similarity Score (OSS vs CEISA)', fontsize=13, fontweight='bold')

    # Histogram
    ax1 = axes[0]
    ax1.hist(df_fuzzy['composite_score'], bins=30, color='#2196F3', edgecolor='white')
    ax1.axvline(LOWER_THRESHOLD, color='orange', linestyle='--', linewidth=2, label=f'Lower {LOWER_THRESHOLD}')
    ax1.axvline(UPPER_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Upper {UPPER_THRESHOLD}')
    ax1.set_xlabel('Composite Score')
    ax1.set_ylabel('Frekuensi')
    ax1.set_title('Histogram Composite Score')
    ax1.legend()

    # CDF
    ax2 = axes[1]
    scores_sorted = np.sort(df_fuzzy['composite_score'])
    cdf = np.arange(1, len(scores_sorted) + 1) / len(scores_sorted)
    ax2.plot(scores_sorted, cdf, color='#4CAF50', linewidth=2)
    ax2.axvline(LOWER_THRESHOLD, color='orange', linestyle='--', linewidth=1.5, label=f'Lower {LOWER_THRESHOLD}')
    ax2.axvline(UPPER_THRESHOLD, color='red', linestyle='--', linewidth=1.5, label=f'Upper {UPPER_THRESHOLD}')
    ax2.set_xlabel('Composite Score')
    ax2.set_ylabel('CDF')
    ax2.set_title('Cumulative Distribution')
    ax2.legend()

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'   Chart disimpan: {output_path}')


def threshold_analysis(df_fuzzy):
    print('\n   Analisis dampak threshold:')
    print(f'   {"Klasifikasi":<14} {"Jumlah":>8}  {"Persen":>8}')
    print('   ' + '-' * 35)
    if df_fuzzy.empty:
        print('   (tidak ada kandidat fuzzy)')
        return
    counts = df_fuzzy['match_type'].value_counts()
    for klas in ['FUZZY_MATCH', 'FUZZY_REVIEW', 'NON_MATCH']:
        n = counts.get(klas, 0)
        pct = n / len(df_fuzzy) * 100
        print(f'   {klas:<14} {n:>8,}  {pct:>7.1f}%')


# ============================================================
# LANGKAH 8: MANUAL SPOT-CHECK
# ============================================================

def spot_check(df_fuzzy, n_sample=3):
    print('\n   Manual spot-check per zona:')
    if df_fuzzy.empty:
        print('   (tidak ada kandidat fuzzy)')
        return

    for zona in ['FUZZY_MATCH', 'FUZZY_REVIEW', 'NON_MATCH']:
        sample = df_fuzzy[df_fuzzy['match_type'] == zona].head(n_sample)
        n_total = (df_fuzzy['match_type'] == zona).sum()
        print(f'\n   --- Zona {zona} ({n_total:,} pairs) ---')
        for _, row in sample.iterrows():
            print(f"     [{row['composite_score']:.4f}] OSS: {row['nama_oss']!r}  <->  CEISA: {row['nama_ceisa']!r} "
                  f"(sim_npwp={row['sim_npwp']:.2f}, sim_nama={row['sim_nama']:.2f}, sim_alamat={row['sim_alamat']:.2f})")


# ============================================================
# LANGKAH 9: DUPLICATE CLUSTERING (INTERNAL PER SUMBER)
# ============================================================

def cluster_oss_duplicates(df_oss):
    """Cluster record OSS dengan nib_digits sama (anomali 'Duplicate Entry')."""
    G = nx.Graph()
    G.add_nodes_from(df_oss.index)

    groups = df_oss[df_oss['nib_digits'] != DUMMY_NIB].groupby('nib_digits').groups
    for nib_digits, idxs in groups.items():
        idxs = list(idxs)
        if len(idxs) < 2:
            continue
        for i in range(len(idxs) - 1):
            G.add_edge(idxs[i], idxs[i + 1])

    components = [c for c in nx.connected_components(G) if len(c) > 1]

    rows = []
    for cid, comp in enumerate(components, 1):
        for idx in comp:
            rows.append({
                'cluster_id': f'OSS_{cid}',
                'source': 'OSS',
                'record_id': idx,
                'NIB': df_oss.loc[idx, 'NIB'],
                'nama': df_oss.loc[idx, 'NAMA_PERSEROAN'],
            })
    return pd.DataFrame(rows, columns=['cluster_id', 'source', 'record_id', 'NIB', 'nama'])


def cluster_ceisa_similar(df_ceisa):
    """Cluster record CEISA dengan nama/alamat sangat mirip TAPI NIB berbeda
    (dedup by NIB sudah selesai di Tahap 2 - fokus ke kandidat lain)."""
    idx_df = df_ceisa.set_index('ID_PERUSAHAAN')

    indexer = recordlinkage.Index()
    indexer.sortedneighbourhood('nama_key', window=5)
    candidate_links = indexer.index(idx_df)

    G = nx.Graph()
    G.add_nodes_from(idx_df.index)

    for id1, id2 in candidate_links:
        r1 = idx_df.loc[id1]
        r2 = idx_df.loc[id2]
        if r1['nib_digits'] == r2['nib_digits']:
            continue  # NIB sama -> sudah ditangani Tahap 2 (Data Mart Dedup)

        sim_nama = fuzz.token_set_ratio(r1['nama_key'], r2['nama_key']) / 100
        sim_alamat = fuzz.token_set_ratio(r1['alamat_key'], r2['alamat_key']) / 100
        composite = sim_nama * 0.6 + sim_alamat * 0.4

        if composite >= UPPER_THRESHOLD:
            G.add_edge(id1, id2)

    components = [c for c in nx.connected_components(G) if len(c) > 1]

    rows = []
    for cid, comp in enumerate(components, 1):
        for record_id in comp:
            rows.append({
                'cluster_id': f'CEISA_{cid}',
                'source': 'CEISA',
                'record_id': record_id,
                'NIB': idx_df.loc[record_id, 'NIB'],
                'nama': idx_df.loc[record_id, 'NAMA_PERUSAHAAN'],
            })
    return pd.DataFrame(rows, columns=['cluster_id', 'source', 'record_id', 'NIB', 'nama'])


# ============================================================
# MAIN PIPELINE (Langkah 1-10)
# ============================================================



# TAHAP 4: GOLDEN RECORD & SURVIVORSHIP

**Tujuan:** Membentuk satu Golden Record per entitas dari gabungan OSS dan CEISA.

### Penjelasan

Aturan survivorship:
- OSS menang untuk data legalitas (System of Record)
- CEISA menang untuk data operasional
- Orphan records tetap masuk dengan SOURCE=OSS_ONLY/CEISA_ONLY

In [ ]:
# [TAHAP 4] GOLDEN RECORD & SURVIVORSHIP
# Berdasarkan: docs/tahapan/tahap4_golden_record.md & docs/02_business_rules.md §3
#
# Untuk setiap pasangan match dari Tahap 3 (EXACT_NIB, EXACT_NPWP, FUZZY_MATCH/REVIEW),
# gabungkan record OSS + CEISA menjadi satu Golden Record menggunakan survivorship
# rules (§3.A): Legalitas & Status -> OSS menang (System of Record), Operasional
# (KODE_KANTOR, NOMOR_TELPON, dll) -> CEISA menang. Record yang tidak match (orphan)
# tetap masuk apa adanya dengan SOURCE=OSS_ONLY / CEISA_ONLY.

import pandas as pd
import numpy as np
import re
from datetime import datetime
# fuzz already imported
# extract_digits, normalize_for_matching, DUMMY_NIB already defined
# STATUS_NIB_POOL already defined

NPWP_PATTERN = re.compile(r'^\d{2}\.\d{3}\.\d{3}\.\d{1}-\d{3}\.\d{3}$')
NIB_PATTERN = re.compile(r'^\d{13}$')


# ============================================================
# LANGKAH 2: SURVIVORSHIP RULES (business_rules §3.A)
# ============================================================
# field_golden -> (kolom_OSS, kolom_CEISA, sumber_menang)
#   'OSS'        : field ada di kedua sumber -> OSS menang (Legalitas & Status / System of Record)
#   'OSS_ONLY'   : field hanya tersedia di OSS (legalitas/perizinan)
#   'CEISA_ONLY' : field hanya tersedia di CEISA (operasional, termasuk KODE_KANTOR & NOMOR_TELPON)

FIELD_RULES = {
    'NIB':                ('NIB', 'NIB', 'OSS'),
    'NPWP':               ('NPWP_PERSEROAN', 'NPWP', 'OSS'),
    'NAMA':               ('NAMA_PERSEROAN', 'NAMA_PERUSAHAAN', 'OSS'),
    'NAMA_SINGKATAN':     ('NAMA_SINGKATAN', None, 'OSS_ONLY'),
    'JENIS_PERSEROAN':    ('JENIS_PERSEROAN', None, 'OSS_ONLY'),
    'STATUS_BADAN_HUKUM': ('STATUS_BADAN_HUKUM', None, 'OSS_ONLY'),
    'STATUS_PERSEROAN':   ('STATUS_PERSEROAN', None, 'OSS_ONLY'),
    'ALAMAT':             ('ALAMAT_PERSEROAN', 'ALAMAT_PERUSAHAAN', 'OSS'),
    'KELURAHAN':          ('KELURAHAN_PERSEROAN', 'KELURAHAN', 'OSS'),
    'DAERAH_ID':          ('PERSEROAN_DAERAH_ID', 'DAERAH_ID', 'OSS'),
    'KODE_POS':           ('KODE_POS_PERSEROAN', 'KODE_POS', 'OSS'),
    'FLAG_IMPOR':         ('FLAG_IMPOR', None, 'OSS_ONLY'),
    'FLAG_EKSPOR':        ('FLAG_EKSPOR', None, 'OSS_ONLY'),
    'JENIS_API':          ('JENIS_API', None, 'OSS_ONLY'),
    'KODE_KANTOR':        (None, 'KODE_KANTOR', 'CEISA_ONLY'),
    'NOMOR_TELPON':       (None, 'NOMOR_TELPON', 'CEISA_ONLY'),
    'KATEGORI':           (None, 'KATEGORI', 'CEISA_ONLY'),
    'NIPER':              (None, 'NIPER', 'CEISA_ONLY'),
    'NOMOR_API':          (None, 'NOMOR_API', 'CEISA_ONLY'),
    'ID_PERUSAHAAN':      (None, 'ID_PERUSAHAAN', 'CEISA_ONLY'),
    'TGL_PERUBAHAN_NIB':  ('TGL_PERUBAHAN_NIB', None, 'OSS_ONLY'),
    'TGL_TERBIT_NIB':     (None, 'TGL_TERBIT_NIB', 'CEISA_ONLY'),
    'TGL_SYNC_OSS':       (None, 'TGL_SYNC_OSS', 'CEISA_ONLY'),
    'STATUS_NIB':         ('STATUS_NIB', 'STATUS_NIB', 'OSS'),
}

META_COLS = [
    'SOURCE', 'MATCH_TYPE', 'SOURCE_COUNT', 'N_CONFLICTS',
    'IS_OUT_OF_SYNC', 'IS_STALE', 'HIGH_SYNC_LAG', 'IS_LOGICAL_CONFLICT_NIPER',
    'CREATED_AT',
]
GOLDEN_COLUMNS = ['GR_ID'] + list(FIELD_RULES.keys()) + META_COLS


# ============================================================
# LANGKAH 3: FUNGSI create_golden_record()
# ============================================================

def create_golden_record(oss_row, ceisa_row, field_rules):
    """Gabungkan satu pasangan (oss_row, ceisa_row) menjadi satu golden record
    berdasarkan field_rules. oss_row/ceisa_row boleh None untuk orphan record."""
    golden = {}
    provenance = {}

    for field, (oss_col, ceisa_col, winner) in field_rules.items():
        oss_val = oss_row[oss_col] if (oss_row is not None and oss_col is not None) else None
        ceisa_val = ceisa_row[ceisa_col] if (ceisa_row is not None and ceisa_col is not None) else None

        if winner == 'CEISA_ONLY':
            if pd.notna(ceisa_val) and str(ceisa_val).strip() != '':
                value, source = ceisa_val, 'CEISA'
            else:
                value, source = None, 'N/A'
        elif winner == 'OSS_ONLY':
            if pd.notna(oss_val) and str(oss_val).strip() != '':
                value, source = oss_val, 'OSS'
            else:
                value, source = None, 'N/A'
        else:  # 'OSS' -> field ada di kedua sumber, OSS menang (System of Record)
            if pd.notna(oss_val) and str(oss_val).strip() != '':
                value, source = oss_val, 'OSS'
            elif pd.notna(ceisa_val) and str(ceisa_val).strip() != '':
                value, source = ceisa_val, 'CEISA'
            else:
                value, source = None, 'N/A'

        golden[field] = value
        provenance[field] = source

    return golden, provenance


def detect_field_conflicts(oss_row, ceisa_row):
    """Bandingkan field yang ada di kedua sumber, kembalikan list konflik
    (field, nilai_oss, nilai_ceisa). Dipakai untuk Langkah 6 - Analisis Pola Konflik."""
    conflicts = []

    # STATUS_NIB - target anomali "Sync Conflict" (tahap0 §4)
    if str(oss_row['STATUS_NIB']).strip() != str(ceisa_row['STATUS_NIB']).strip():
        conflicts.append(('STATUS_NIB', oss_row['STATUS_NIB'], ceisa_row['STATUS_NIB']))

    # NPWP - bedakan konflik NILAI (digit beda) vs konflik FORMAT (digit sama, format beda)
    d_oss, d_ceisa = extract_digits(oss_row['NPWP_PERSEROAN']), extract_digits(ceisa_row['NPWP'])
    if d_oss != d_ceisa:
        conflicts.append(('NPWP_VALUE', oss_row['NPWP_PERSEROAN'], ceisa_row['NPWP']))
    elif str(oss_row['NPWP_PERSEROAN']) != str(ceisa_row['NPWP']):
        conflicts.append(('NPWP_FORMAT', oss_row['NPWP_PERSEROAN'], ceisa_row['NPWP']))

    # NAMA - target anomali "Fuzzy Identity"
    if normalize_for_matching(oss_row['NAMA_PERSEROAN']) != normalize_for_matching(ceisa_row['NAMA_PERUSAHAAN']):
        conflicts.append(('NAMA', oss_row['NAMA_PERSEROAN'], ceisa_row['NAMA_PERUSAHAAN']))

    # ALAMAT
    if normalize_for_matching(oss_row['ALAMAT_PERSEROAN']) != normalize_for_matching(ceisa_row['ALAMAT_PERUSAHAAN']):
        conflicts.append(('ALAMAT', oss_row['ALAMAT_PERSEROAN'], ceisa_row['ALAMAT_PERUSAHAAN']))

    # KELURAHAN, DAERAH_ID, KODE_POS - bandingkan jika kedua sisi terisi
    for oss_col, ceisa_col, field_name in [
        ('KELURAHAN_PERSEROAN', 'KELURAHAN', 'KELURAHAN'),
        ('PERSEROAN_DAERAH_ID', 'DAERAH_ID', 'DAERAH_ID'),
        ('KODE_POS_PERSEROAN', 'KODE_POS', 'KODE_POS'),
    ]:
        v_oss, v_ceisa = oss_row[oss_col], ceisa_row[ceisa_col]
        if pd.notna(v_oss) and pd.notna(v_ceisa) and str(v_oss).strip() != str(v_ceisa).strip():
            conflicts.append((field_name, v_oss, v_ceisa))

    return conflicts


def is_logical_conflict_niper(oss_row, ceisa_row):
    """Anomali 'Logical Conflict': FLAG_EKSPOR (OSS) = 'N' tapi NIPER (CEISA) terisi."""
    niper = ceisa_row['NIPER']
    niper_filled = pd.notna(niper) and str(niper).strip() not in ('', 'nan')
    return bool(oss_row['FLAG_EKSPOR'] == 'N' and niper_filled)


# ============================================================
# LANGKAH 4: PROSES MATCHED PAIRS (batch generation)
# ============================================================

def lookup_oss_row(oss_idx, nib, ceisa_row):
    """Ambil baris OSS berdasarkan NIB. NIB dummy "0000000000000" (anomali NIB Invalid)
    dimiliki banyak baris berbeda -> disambiguasi dengan kemiripan nama ke baris CEISA pasangannya."""
    candidates = oss_idx.loc[[nib]]
    if len(candidates) == 1:
        return candidates.iloc[0]
    # NIB dummy "0000000000000" -> banyak baris dengan index label sama, gunakan posisi (iloc)
    scores = candidates['NAMA_PERSEROAN'].apply(
        lambda nama: fuzz.token_set_ratio(normalize_for_matching(nama), normalize_for_matching(ceisa_row['NAMA_PERUSAHAAN']))
    )
    return candidates.iloc[scores.values.argmax()]


def build_matched_records(df_oss, df_ceisa, candidate_pairs, field_rules):
    # Dedup residu anomali "Duplicate Entry" - 1 NIB OSS hanya 1 baris untuk lookup
    pairs = candidate_pairs.drop_duplicates(subset='NIB_OSS', keep='first')

    oss_idx = df_oss.set_index('NIB', drop=False)
    ceisa_idx = df_ceisa.set_index('ID_PERUSAHAAN', drop=False)

    golden_rows, provenance_rows, conflict_rows = [], [], []

    matched_oss_row_ids = set()

    for _, pair in pairs.iterrows():
        ceisa_row = ceisa_idx.loc[pair['ID_PERUSAHAAN_CEISA']]
        oss_row = lookup_oss_row(oss_idx, pair['NIB_OSS'], ceisa_row)
        matched_oss_row_ids.add(oss_row['_ROW_ID'])

        golden, provenance = create_golden_record(oss_row, ceisa_row, field_rules)

        is_out_of_sync = str(oss_row['STATUS_NIB']).strip() != str(ceisa_row['STATUS_NIB']).strip()
        is_logic_conflict = is_logical_conflict_niper(oss_row, ceisa_row)
        field_conflicts = detect_field_conflicts(oss_row, ceisa_row)
        n_conflicts = len(field_conflicts) + (1 if is_logic_conflict else 0)

        golden.update({
            'SOURCE': 'MATCHED',
            'MATCH_TYPE': pair['match_type'],
            'SOURCE_COUNT': 2,
            'N_CONFLICTS': n_conflicts,
            'IS_OUT_OF_SYNC': is_out_of_sync,          # §3.B - anomali "Sync Conflict"
            'IS_STALE': bool(oss_row['IS_STALE']),     # dibawa dari Tahap 1/2, tidak dihitung ulang
            'HIGH_SYNC_LAG': bool(ceisa_row['HIGH_SYNC_LAG']),
            'IS_LOGICAL_CONFLICT_NIPER': is_logic_conflict,
            'CREATED_AT': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        })
        golden_rows.append(golden)

        for field, source in provenance.items():
            provenance_rows.append({'NIB': golden['NIB'], 'field': field, 'source': source})

        for field, oss_val, ceisa_val in field_conflicts:
            conflict_rows.append({
                'NIB': golden['NIB'], 'field': field,
                'oss_value': oss_val, 'ceisa_value': ceisa_val, 'winner': 'OSS',
            })
        if is_logic_conflict:
            conflict_rows.append({
                'NIB': golden['NIB'], 'field': 'FLAG_EKSPOR_vs_NIPER',
                'oss_value': oss_row['FLAG_EKSPOR'], 'ceisa_value': ceisa_row['NIPER'], 'winner': 'OSS (FLAG_EKSPOR)',
            })

    matched_ceisa_ids = set(pairs['ID_PERUSAHAAN_CEISA'])
    return golden_rows, provenance_rows, conflict_rows, matched_oss_row_ids, matched_ceisa_ids


# ============================================================
# LANGKAH 5: PROSES ORPHAN RECORDS (OSS_ONLY / CEISA_ONLY)
# ============================================================

def build_orphan_records(df_oss, df_ceisa, matched_oss_row_ids, matched_ceisa_ids, field_rules):
    golden_rows, provenance_rows = [], []
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    df_oss_orphan = df_oss[~df_oss['_ROW_ID'].isin(matched_oss_row_ids)]
    for _, oss_row in df_oss_orphan.iterrows():
        golden, provenance = create_golden_record(oss_row, None, field_rules)
        golden.update({
            'SOURCE': 'OSS_ONLY', 'MATCH_TYPE': 'N/A', 'SOURCE_COUNT': 1, 'N_CONFLICTS': 0,
            'IS_OUT_OF_SYNC': False, 'IS_STALE': bool(oss_row['IS_STALE']),
            'HIGH_SYNC_LAG': None, 'IS_LOGICAL_CONFLICT_NIPER': False, 'CREATED_AT': timestamp,
        })
        golden_rows.append(golden)
        for field, source in provenance.items():
            provenance_rows.append({'NIB': golden['NIB'], 'field': field, 'source': source})

    df_ceisa_orphan = df_ceisa[~df_ceisa['ID_PERUSAHAAN'].isin(matched_ceisa_ids)]
    for _, ceisa_row in df_ceisa_orphan.iterrows():
        golden, provenance = create_golden_record(None, ceisa_row, field_rules)
        golden.update({
            'SOURCE': 'CEISA_ONLY', 'MATCH_TYPE': 'N/A', 'SOURCE_COUNT': 1, 'N_CONFLICTS': 0,
            'IS_OUT_OF_SYNC': False, 'IS_STALE': None,
            'HIGH_SYNC_LAG': bool(ceisa_row['HIGH_SYNC_LAG']), 'IS_LOGICAL_CONFLICT_NIPER': False, 'CREATED_AT': timestamp,
        })
        golden_rows.append(golden)
        for field, source in provenance.items():
            provenance_rows.append({'NIB': golden['NIB'], 'field': field, 'source': source})

    return golden_rows, provenance_rows


# ============================================================
# LANGKAH 6: ANALISIS POLA KONFLIK
# ============================================================

def analyze_conflict_patterns(df_conflicts, df_golden):
    n_matched = (df_golden['SOURCE'] == 'MATCHED').sum()
    print(f'\n   Total matched pairs       : {n_matched:,}')
    print(f'   Pairs dengan >=1 konflik   : {(df_golden.loc[df_golden["SOURCE"] == "MATCHED", "N_CONFLICTS"] > 0).sum():,}')

    if df_conflicts.empty:
        print('   (tidak ada konflik field terdeteksi)')
        return

    by_field = df_conflicts['field'].value_counts().reset_index()
    by_field.columns = ['field', 'jumlah_konflik']
    by_field['persen_dari_matched'] = (by_field['jumlah_konflik'] / n_matched * 100).round(2)

    print('\n   Frekuensi konflik per field:')
    for _, row in by_field.iterrows():
        print(f'   {row["field"]:<22} {row["jumlah_konflik"]:>6,}  ({row["persen_dari_matched"]:>5.1f}% dari matched pairs)')

    n_out_of_sync = df_golden['IS_OUT_OF_SYNC'].sum()
    n_logic = df_golden['IS_LOGICAL_CONFLICT_NIPER'].sum()
    print(f'\n   IS_OUT_OF_SYNC (STATUS_NIB beda)        : {n_out_of_sync:,} record')
    print(f'   IS_LOGICAL_CONFLICT (FLAG_EKSPOR vs NIPER): {n_logic:,} record')


# ============================================================
# LANGKAH 7: PROVENANCE ANALYSIS
# ============================================================

def provenance_analysis(df_provenance):
    summary = df_provenance.groupby(['field', 'source']).size().unstack(fill_value=0)
    for src in ['OSS', 'CEISA', 'N/A']:
        if src not in summary.columns:
            summary[src] = 0
    summary = summary[['OSS', 'CEISA', 'N/A']]
    pct = summary.div(summary.sum(axis=1), axis=0) * 100

    print('\n   Distribusi sumber per field (jumlah record):')
    print(summary.to_string())

    overall = df_provenance['source'].value_counts(normalize=True) * 100
    print('\n   Kontribusi keseluruhan per sumber:')
    for src, val in overall.items():
        bar = '#' * int(val / 2)
        print(f'   {src:<6}: {val:5.1f}%  {bar}')

    return summary, pct


# ============================================================
# LANGKAH 8: QUALITY VALIDATION GOLDEN RECORD
# ============================================================

def validate_golden_record(df_golden):
    print('\n   Quality checks:')
    checks = []

    # NIB dummy "0000000000000" (anomali NIB Invalid - Tahap 1 Validitas) secara sah dimiliki
    # banyak perusahaan berbeda -> dikecualikan dari cek keunikan NIB.
    is_dummy = df_golden['NIB'] == DUMMY_NIB
    n_dup_dummy = is_dummy.sum()
    n_dup_other = df_golden.loc[~is_dummy, 'NIB'].duplicated().sum()
    checks.append(('NIB unik (di luar NIB dummy/invalid)', n_dup_other == 0, f'{n_dup_other:,} NIB duplikat'))

    n_nib_invalid = (~df_golden['NIB'].astype(str).str.match(NIB_PATTERN)).sum()
    checks.append(('Format NIB (13 digit)', n_nib_invalid == 0, f'{n_nib_invalid:,} NIB format tidak valid'))

    n_npwp_invalid = (~df_golden['NPWP'].astype(str).str.match(NPWP_PATTERN)).sum()
    checks.append(('Format NPWP (XX.XXX.XXX.X-XXX.XXX)', n_npwp_invalid == 0, f'{n_npwp_invalid:,} NPWP format tidak valid'))

    n_status_invalid = (~df_golden['STATUS_NIB'].isin(STATUS_NIB_POOL)).sum()
    checks.append(('STATUS_NIB valid (AKTIF/DIBEKUKAN/DICABUT)', n_status_invalid == 0, f'{n_status_invalid:,} STATUS_NIB tidak valid'))

    for name, passed, detail in checks:
        status = 'PASS' if passed else 'INFO'
        print(f'   [{status}] {name:<42} | {detail}')

    if n_dup_other > 0:
        print(f'\n   -> {n_dup_other:,} NIB duplikat (di luar dummy) di-drop (keep first).')
        keep_mask = is_dummy | ~df_golden['NIB'].duplicated()
        df_golden = df_golden[keep_mask].reset_index(drop=True)

    if n_dup_dummy > 0:
        print(f'\n   INFO: {n_dup_dummy:,} golden record memiliki NIB dummy "{DUMMY_NIB}"')
        print('         (anomali NIB Invalid "Dummy/Kosong") - tetap dipertahankan sebagai record')
        print('         terpisah (GR_ID unik) karena merepresentasikan perusahaan berbeda.')

    return df_golden


# ============================================================
# MAIN PIPELINE (Langkah 1-9)
# ============================================================



# TAHAP 5: DATA QUALITY MONITORING

**Tujuan:** Mengukur dan membandingkan kualitas data Before vs After MDM.

### Penjelasan

5 dimensi DMBOK: Completeness, Validity, Uniqueness, Consistency, Timeliness.
Dijalankan pada 3 dataset: OSS, CEISA, Golden Record.

In [ ]:
# [TAHAP 5] DATA QUALITY MONITORING
# Berdasarkan: docs/tahapan/tahap5_dq_monitoring.md & docs/02_business_rules.md §1, §4
#
# Quality Rules Engine sederhana (rule_id, dimension, field, check, severity) dijalankan
# pada 3 dataset: OSS (before), CEISA (before), Golden Record (after) - lalu dibandingkan
# dalam satu scorecard per dimensi DMBOK (Completeness, Validity, Uniqueness, Consistency,
# Timeliness) untuk menunjukkan peningkatan kualitas data hasil MDM.

import pandas as pd
import re
from dataclasses import dataclass
from typing import Callable

# Already defined above
# DUMMY_NIB already defined

NIB_PATTERN = re.compile(r'^\d{13}$')
NPWP_PATTERN = re.compile(r'^\d{2}\.\d{3}\.\d{3}\.\d{1}-\d{3}\.\d{3}$')
KODE_POS_PATTERN = re.compile(r'^\d{5}$')

DIMENSIONS = ['Completeness', 'Validity', 'Uniqueness', 'Consistency', 'Timeliness']


@dataclass
class QualityRule:
    rule_id: str
    dimension: str
    field: str
    description: str
    check: Callable[[pd.DataFrame], pd.Series]  # df -> bool Series (True = pass)
    severity: str  # HIGH / MEDIUM / LOW


@dataclass
class RuleResult:
    dataset: str
    rule_id: str
    dimension: str
    field: str
    description: str
    severity: str
    total: int
    n_pass: int
    n_fail: int
    pass_rate: float


# --- Quality Rules Engine: check builders (Langkah 2) ---

def _to_digit_str(series):
    """KODE_POS terbaca float64 (mis. 1330.0) - konversi ke string digit tanpa '.0'
    agar panjang digit (utk cek format 5-digit) tetap apa adanya (leading zero yang
    hilang akibat tipe numerik akan terdeteksi sebagai format tidak valid)."""
    def conv(x):
        if pd.isna(x):
            return None
        if isinstance(x, float) and x.is_integer():
            return str(int(x))
        return str(x)
    return series.apply(conv)


def check_completeness(field):
    """Field wajib (NIB/NPWP/NAMA/STATUS_NIB) tidak boleh null/kosong."""
    def _check(df):
        col = df[field]
        return col.notna() & (col.astype(str).str.strip() != '')
    return _check


def check_format(field, pattern, numeric=False):
    """Field opsional/format: nilai kosong dianggap PASS (itu isu Completeness, bukan
    Validity), nilai yang terisi harus cocok dengan pattern."""
    def _check(df):
        col = _to_digit_str(df[field]) if numeric else df[field]
        is_null = col.isna()
        match = col.astype(str).str.match(pattern)
        return is_null | match
    return _check


def check_not_dummy_nib(df):
    """NIB tidak boleh nilai dummy/kosong "0000000000000" (anomali NIB Invalid - Dummy/Kosong)."""
    return df['NIB'].astype(str) != DUMMY_NIB


def check_enum(field, pool):
    """Nilai field (jika terisi) harus termasuk dalam pool referensi yang sah."""
    def _check(df):
        col = df[field]
        return col.isna() | col.isin(pool)
    return _check


def check_unique_nib(df):
    """NIB unik secara internal - kecuali NIB dummy "0000000000000" yang secara sah
    dimiliki banyak entitas berbeda (lihat tahap4_golden_record.md langkah 8)."""
    nib = df['NIB'].astype(str)
    is_dummy = nib == DUMMY_NIB
    is_dup = nib.duplicated(keep=False)
    return ~(is_dup & ~is_dummy)


def check_flag_impor_jenis_api(df):
    """FLAG_IMPOR='Y' -> JENIS_API harus terisi; FLAG_IMPOR='N' -> JENIS_API harus kosong."""
    flag = df['FLAG_IMPOR']
    jenis_filled = df['JENIS_API'].notna() & (df['JENIS_API'].astype(str).str.strip() != '')
    return flag.isna() | ((flag == 'Y') & jenis_filled) | ((flag == 'N') & ~jenis_filled)


def check_kategori_niper(df):
    """KATEGORI='IMPORTIR' -> NIPER harus kosong; KATEGORI EKSPORTIR/KEDUA-DUANYA -> NIPER
    harus terisi (anomali "Logical Conflict")."""
    kategori = df['KATEGORI']
    niper_filled = df['NIPER'].notna() & (df['NIPER'].astype(str).str.strip() != '')
    return (
        kategori.isna()
        | ((kategori == 'IMPORTIR') & ~niper_filled)
        | (kategori.isin(['EKSPORTIR', 'KEDUA-DUANYA']) & niper_filled)
    )


def check_flag_is_false(field):
    """Flag boolean (IS_STALE/HIGH_SYNC_LAG/IS_OUT_OF_SYNC/IS_LOGICAL_CONFLICT_NIPER) harus
    False. NaN (flag tidak relevan utk record ybs, mis. OSS_ONLY tanpa HIGH_SYNC_LAG)
    dianggap PASS (tidak berlaku)."""
    def _check(df):
        col = df[field].apply(lambda x: bool(x) if pd.notna(x) else False)
        return ~col
    return _check


# --- Definisi rules per dataset (Langkah 3) ---

OSS_RULES = [
    QualityRule('COMP-NIB', 'Completeness', 'NIB', 'NIB tidak boleh kosong',
                check_completeness('NIB'), 'HIGH'),
    QualityRule('COMP-NPWP', 'Completeness', 'NPWP_PERSEROAN', 'NPWP tidak boleh kosong',
                check_completeness('NPWP_PERSEROAN'), 'HIGH'),
    QualityRule('COMP-NAMA', 'Completeness', 'NAMA_PERSEROAN', 'NAMA tidak boleh kosong',
                check_completeness('NAMA_PERSEROAN'), 'HIGH'),
    QualityRule('COMP-STATUS_NIB', 'Completeness', 'STATUS_NIB', 'STATUS_NIB tidak boleh kosong',
                check_completeness('STATUS_NIB'), 'HIGH'),

    QualityRule('VAL-NIB-FORMAT', 'Validity', 'NIB', 'NIB harus 13 digit numerik',
                check_format('NIB', NIB_PATTERN), 'HIGH'),
    QualityRule('VAL-NIB-DUMMY', 'Validity', 'NIB', 'NIB tidak boleh dummy "0000000000000"',
                check_not_dummy_nib, 'MEDIUM'),
    QualityRule('VAL-NPWP-FORMAT', 'Validity', 'NPWP_PERSEROAN', 'NPWP harus format XX.XXX.XXX.X-XXX.XXX',
                check_format('NPWP_PERSEROAN', NPWP_PATTERN), 'HIGH'),
    QualityRule('VAL-KODE_POS-FORMAT', 'Validity', 'KODE_POS_PERSEROAN', 'KODE_POS (jika terisi) harus 5 digit',
                check_format('KODE_POS_PERSEROAN', KODE_POS_PATTERN, numeric=True), 'LOW'),
    QualityRule('VAL-STATUS_NIB-ENUM', 'Validity', 'STATUS_NIB', f'STATUS_NIB harus salah satu dari {STATUS_NIB_POOL}',
                check_enum('STATUS_NIB', STATUS_NIB_POOL), 'HIGH'),
    QualityRule('VAL-JENIS_PERSEROAN-ENUM', 'Validity', 'JENIS_PERSEROAN', f'JENIS_PERSEROAN harus salah satu dari {JENIS_PERSEROAN_POOL}',
                check_enum('JENIS_PERSEROAN', JENIS_PERSEROAN_POOL), 'MEDIUM'),

    QualityRule('UNIQ-NIB', 'Uniqueness', 'NIB', 'NIB unik (di luar NIB dummy)',
                check_unique_nib, 'HIGH'),

    QualityRule('CONS-FLAG_IMPOR-JENIS_API', 'Consistency', 'FLAG_IMPOR/JENIS_API',
                'FLAG_IMPOR konsisten dengan pengisian JENIS_API',
                check_flag_impor_jenis_api, 'MEDIUM'),

    QualityRule('TIME-IS_STALE', 'Timeliness', 'IS_STALE', 'TGL_PERUBAHAN_NIB tidak boleh > 1 tahun (IS_STALE)',
                check_flag_is_false('IS_STALE'), 'MEDIUM'),
]

CEISA_RULES = [
    QualityRule('COMP-NIB', 'Completeness', 'NIB', 'NIB tidak boleh kosong',
                check_completeness('NIB'), 'HIGH'),
    QualityRule('COMP-NPWP', 'Completeness', 'NPWP', 'NPWP tidak boleh kosong',
                check_completeness('NPWP'), 'HIGH'),
    QualityRule('COMP-NAMA', 'Completeness', 'NAMA_PERUSAHAAN', 'NAMA tidak boleh kosong',
                check_completeness('NAMA_PERUSAHAAN'), 'HIGH'),
    QualityRule('COMP-STATUS_NIB', 'Completeness', 'STATUS_NIB', 'STATUS_NIB tidak boleh kosong',
                check_completeness('STATUS_NIB'), 'HIGH'),

    QualityRule('VAL-NIB-FORMAT', 'Validity', 'NIB', 'NIB harus 13 digit numerik',
                check_format('NIB', NIB_PATTERN), 'HIGH'),
    QualityRule('VAL-NIB-DUMMY', 'Validity', 'NIB', 'NIB tidak boleh dummy "0000000000000"',
                check_not_dummy_nib, 'MEDIUM'),
    QualityRule('VAL-NPWP-FORMAT', 'Validity', 'NPWP', 'NPWP harus format XX.XXX.XXX.X-XXX.XXX (anomali Inconsistent NPWP)',
                check_format('NPWP', NPWP_PATTERN), 'HIGH'),
    QualityRule('VAL-KODE_POS-FORMAT', 'Validity', 'KODE_POS', 'KODE_POS (jika terisi) harus 5 digit',
                check_format('KODE_POS', KODE_POS_PATTERN, numeric=True), 'LOW'),
    QualityRule('VAL-STATUS_NIB-ENUM', 'Validity', 'STATUS_NIB', f'STATUS_NIB harus salah satu dari {STATUS_NIB_POOL}',
                check_enum('STATUS_NIB', STATUS_NIB_POOL), 'HIGH'),
    QualityRule('VAL-KATEGORI-ENUM', 'Validity', 'KATEGORI', f'KATEGORI harus salah satu dari {KATEGORI_CEISA_POOL}',
                check_enum('KATEGORI', KATEGORI_CEISA_POOL), 'MEDIUM'),

    QualityRule('UNIQ-NIB', 'Uniqueness', 'NIB', 'NIB unik (1 baris per NIB - data mart sudah di-dedup Tahap 2)',
                check_unique_nib, 'HIGH'),

    QualityRule('CONS-KATEGORI-NIPER', 'Consistency', 'KATEGORI/NIPER',
                'KATEGORI konsisten dengan pengisian NIPER (anomali Logical Conflict)',
                check_kategori_niper, 'MEDIUM'),

    QualityRule('TIME-HIGH_SYNC_LAG', 'Timeliness', 'HIGH_SYNC_LAG', 'TGL_SYNC_OSS tidak boleh > 30 hari (HIGH_SYNC_LAG)',
                check_flag_is_false('HIGH_SYNC_LAG'), 'MEDIUM'),
]

GOLDEN_RULES = [
    QualityRule('COMP-NIB', 'Completeness', 'NIB', 'NIB tidak boleh kosong',
                check_completeness('NIB'), 'HIGH'),
    QualityRule('COMP-NPWP', 'Completeness', 'NPWP', 'NPWP tidak boleh kosong',
                check_completeness('NPWP'), 'HIGH'),
    QualityRule('COMP-NAMA', 'Completeness', 'NAMA', 'NAMA tidak boleh kosong',
                check_completeness('NAMA'), 'HIGH'),
    QualityRule('COMP-STATUS_NIB', 'Completeness', 'STATUS_NIB', 'STATUS_NIB tidak boleh kosong',
                check_completeness('STATUS_NIB'), 'HIGH'),

    QualityRule('VAL-NIB-FORMAT', 'Validity', 'NIB', 'NIB harus 13 digit numerik',
                check_format('NIB', NIB_PATTERN), 'HIGH'),
    QualityRule('VAL-NIB-DUMMY', 'Validity', 'NIB', 'NIB tidak boleh dummy "0000000000000"',
                check_not_dummy_nib, 'MEDIUM'),
    QualityRule('VAL-NPWP-FORMAT', 'Validity', 'NPWP', 'NPWP harus format XX.XXX.XXX.X-XXX.XXX',
                check_format('NPWP', NPWP_PATTERN), 'HIGH'),
    QualityRule('VAL-KODE_POS-FORMAT', 'Validity', 'KODE_POS', 'KODE_POS (jika terisi) harus 5 digit',
                check_format('KODE_POS', KODE_POS_PATTERN, numeric=True), 'LOW'),
    QualityRule('VAL-STATUS_NIB-ENUM', 'Validity', 'STATUS_NIB', f'STATUS_NIB harus salah satu dari {STATUS_NIB_POOL}',
                check_enum('STATUS_NIB', STATUS_NIB_POOL), 'HIGH'),
    QualityRule('VAL-JENIS_PERSEROAN-ENUM', 'Validity', 'JENIS_PERSEROAN', f'JENIS_PERSEROAN harus salah satu dari {JENIS_PERSEROAN_POOL}',
                check_enum('JENIS_PERSEROAN', JENIS_PERSEROAN_POOL), 'MEDIUM'),
    QualityRule('VAL-KATEGORI-ENUM', 'Validity', 'KATEGORI', f'KATEGORI harus salah satu dari {KATEGORI_CEISA_POOL}',
                check_enum('KATEGORI', KATEGORI_CEISA_POOL), 'MEDIUM'),

    QualityRule('UNIQ-NIB', 'Uniqueness', 'NIB', 'NIB unik (di luar NIB dummy - lihat tahap4 langkah 8)',
                check_unique_nib, 'HIGH'),

    QualityRule('CONS-FLAG_IMPOR-JENIS_API', 'Consistency', 'FLAG_IMPOR/JENIS_API',
                'FLAG_IMPOR konsisten dengan pengisian JENIS_API',
                check_flag_impor_jenis_api, 'MEDIUM'),
    QualityRule('CONS-KATEGORI-NIPER', 'Consistency', 'KATEGORI/NIPER',
                'KATEGORI konsisten dengan pengisian NIPER (anomali Logical Conflict)',
                check_kategori_niper, 'MEDIUM'),
    QualityRule('CONS-STATUS_NIB-SYNC', 'Consistency', 'IS_OUT_OF_SYNC',
                'STATUS_NIB OSS & CEISA harus sinkron (anomali Sync Conflict)',
                check_flag_is_false('IS_OUT_OF_SYNC'), 'HIGH'),
    QualityRule('CONS-FLAG_EKSPOR-NIPER', 'Consistency', 'IS_LOGICAL_CONFLICT_NIPER',
                'FLAG_EKSPOR konsisten dengan NIPER (anomali Logical Conflict)',
                check_flag_is_false('IS_LOGICAL_CONFLICT_NIPER'), 'HIGH'),

    QualityRule('TIME-IS_STALE', 'Timeliness', 'IS_STALE', 'TGL_PERUBAHAN_NIB tidak boleh > 1 tahun (IS_STALE)',
                check_flag_is_false('IS_STALE'), 'MEDIUM'),
    QualityRule('TIME-HIGH_SYNC_LAG', 'Timeliness', 'HIGH_SYNC_LAG', 'TGL_SYNC_OSS tidak boleh > 30 hari (HIGH_SYNC_LAG)',
                check_flag_is_false('HIGH_SYNC_LAG'), 'MEDIUM'),
]


# --- Eksekusi rules (Langkah 4) ---

def run_rules(df, rules, dataset_name):
    results = []
    total = len(df)
    for rule in rules:
        mask = rule.check(df)
        n_pass = int(mask.sum())
        results.append(RuleResult(
            dataset=dataset_name, rule_id=rule.rule_id, dimension=rule.dimension,
            field=rule.field, description=rule.description, severity=rule.severity,
            total=total, n_pass=n_pass, n_fail=total - n_pass,
            pass_rate=(n_pass / total * 100) if total else 0.0,
        ))
    return results


def print_rule_results(results):
    for r in results:
        status = 'PASS' if r.pass_rate == 100 else ('WARN' if r.pass_rate >= 95 else 'FAIL')
        print(f'   [{status}] {r.rule_id:<26} {r.dimension:<13} | {r.pass_rate:6.2f}% '
              f'({r.n_pass:,}/{r.total:,}) | {r.description}')


# --- Scorecard perbandingan before vs after (Langkah 5) ---

def build_scorecard(df_results):
    pivot = df_results.groupby(['dimension', 'dataset'])['pass_rate'].mean().unstack('dataset')
    pivot = pivot.reindex(DIMENSIONS)[['OSS', 'CEISA', 'GOLDEN']]
    pivot.loc['Overall'] = pivot.mean(axis=0)

    scorecard = pivot.reset_index().rename(columns={
        'dimension': 'dimension', 'OSS': 'oss_score', 'CEISA': 'ceisa_score', 'GOLDEN': 'golden_score',
    })
    scorecard['delta_vs_oss'] = scorecard['golden_score'] - scorecard['oss_score']
    scorecard['delta_vs_ceisa'] = scorecard['golden_score'] - scorecard['ceisa_score']
    return scorecard




# TAHAP 6: YDATA PROFILING DASHBOARD

**Tujuan:** Profiling ulang Golden Record dan perbandingan Before vs After MDM.

### Penjelasan

1. YData Profiling pada Golden Record
2. Perbandingan metrik: jumlah record, missing value, validitas, duplikasi
3. Insight bisnis dan rekomendasi data governance

In [ ]:
# [TAHAP 6] YDATA PROFILING DASHBOARD
# Berdasarkan: docs/tahapan/tahap6_profiling_dashboard.md
#
# Profiling ulang terhadap Golden Record (after MDM) menggunakan ydata_profiling,
# lalu membandingkan ringkasan profiling_before (OSS+CEISA) vs profiling_after
# (Golden Record), ditutup dengan insight bisnis & rekomendasi data governance.

import pandas as pd
from ydata_profiling import ProfileReport

# Already defined above
# from Source.step05_profiling import (
    NIB_PATTERN, NPWP_PATTERN, KODE_POS_PATTERN, _to_digit_str, YDATA_CORRELATIONS,
)

DUMMY_NIB = '0' * 13  # NIB dummy/kosong - sama dengan Source/step07_matching.py


def format_validity_pct(df, nib_col, npwp_col, kp_col):
    """Hitung % valid format untuk NIB, NPWP, dan KODE_POS (dari yang terisi)."""
    nib_valid = df[nib_col].astype(str).str.match(NIB_PATTERN).mean() * 100
    npwp_valid = df[npwp_col].astype(str).str.match(NPWP_PATTERN).mean() * 100

    kp_series = _to_digit_str(df[kp_col])
    kp_filled = kp_series.dropna()
    kp_valid = kp_filled.str.match(KODE_POS_PATTERN).mean() * 100 if len(kp_filled) else 0.0

    return nib_valid, npwp_valid, kp_valid




# SELESAI!

## Pipeline MDM End-to-End Berhasil Dijalankan

**Tahap 0** - Simulasi Data & Faker Engine
**Tahap 1** - Data Profiling (Baseline DQ)
**Tahap 2** - Data Cleansing & Standardization
**Tahap 3** - Duplicate Detection & Matching
**Tahap 4** - Golden Record & Survivorship
**Tahap 5** - Data Quality Monitoring
**Tahap 6** - YData Profiling Dashboard

### Struktur Output

```
/content/
data/
  raw/oss_nib_data.csv, ceisa_data.csv
  processed/oss_cleaned.csv, ceisa_cleaned.csv, dataset_clean.csv
  golden/golden_record.csv
reports/
  profiling_before_oss.html, profiling_before_ceisa.html
  profiling_after.html
  audit_trail.csv, provenance_log.csv, conflict_log.csv
  dq_scorecard.csv, dq_rule_results.csv
  *.png
```

**Mini Project - Master Data Management DJBC**
**Kelompok 5 | Single Importer & Exporter View**